# US Post-Market

## 1. Settings

In [1]:
import io
import logging
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, wait
from datetime import datetime, timedelta, timezone
from functools import partial
from urllib.parse import quote_plus
from zoneinfo import ZoneInfo

import feedparser
import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv
from IPython.display import Markdown, display
from requests.adapters import HTTPAdapter

# 《股市》區塊
INDICES = {"^DJI": "道瓊", "^GSPC": "標普", "^IXIC": "納指", "^SOX": "費半", "^VIX": "VIX"}
ADRS = {"TSM": "台積", "UMC": "聯電", "ASX": "日月光"}           # 台股 ADR
# GICS 類股：維基百科名稱 → (中文, S&P 500 類股指數, 備援 SPDR ETF)
SECTORS = {
    "Information Technology": ("資訊科技", "^SP500-45", "XLK"),
    "Communication Services": ("通訊服務", "^SP500-50", "XLC"),
    "Consumer Discretionary": ("非必需消費", "^SP500-25", "XLY"),
    "Consumer Staples": ("必需消費", "^SP500-30", "XLP"),
    "Energy": ("能源", "^GSPE", "XLE"),             # Yahoo 的能源類股指數代號是 ^GSPE（沒有 ^SP500-10）
    "Financials": ("金融", "^SP500-40", "XLF"),
    "Health Care": ("醫療保健", "^SP500-35", "XLV"),
    "Industrials": ("工業", "^SP500-20", "XLI"),
    "Materials": ("原材料", "^SP500-15", "XLB"),
    "Utilities": ("公用事業", "^SP500-55", "XLU"),
    "Real Estate": ("不動產", "^SP500-60", "XLRE"),
}
FOCUS = {"GS": "Goldman Sachs", "MSFT": "Microsoft", "GOOGL": "Google", "MS": "Morgan Stanley", "AMZN": "Amazon",
         "JPM": "JP Morgan", "META": "Meta", "AAPL": "Apple", "TSLA": "Tesla", "AMD": "AMD", "NVDA": "NVIDIA",
         "SMCI": "SuperMicro"}
ASSETS = {"^TNX": "美債", "DX-Y.NYB": "美元", "GC=F": "黃金", "CL=F": "西德州", "BZ=F": "布蘭特"}
BREADTH = ["RSP", "SPY"]           # 等權重 vs 市值加權
logging.getLogger("yfinance").setLevel(logging.CRITICAL)   # 隱藏 yfinance 的逐檔錯誤訊息，改由第 2 格整理

MOVERS_PER_SECTOR = 3   # 強弱類股各列出幾檔領漲 / 領跌
NEWS_HOURS = 20         # 新聞抓最近幾小時
NEWS_PER_GROUP = 8
SP500_CACHE = "sp500_constituents.csv"
TIMEOUT = 15
YAHOO_WORKERS = 6       # 同時對 Yahoo 發出的請求數（實測 6 沒問題；太多會被限流）
FINNHUB_WORKERS = 8     # Finnhub 免費版：每分鐘 60 次、每秒 30 次
TW = ZoneInfo("Asia/Taipei")
NY = ZoneInfo("America/New_York")
UA = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                     "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")}

# 共用連線：連同一個網站多次時不必每次重新建立 HTTPS 連線
HTTP = requests.Session()
HTTP.headers.update(UA)
HTTP.mount("https://", HTTPAdapter(pool_connections=20, pool_maxsize=20))

# 金鑰放在同資料夾的 .env（不要寫進筆記本）
load_dotenv(".env")
FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY", "").strip()   # 個股新聞、財報、綜合新聞
FRED_API_KEY = os.getenv("FRED_API_KEY", "").strip()         # 美債殖利率備援
print("Finnhub：", "已設定" if FINNHUB_API_KEY else "未設定", "｜FRED：", "已設定" if FRED_API_KEY else "未設定")


def clean(text, limit=200):
    text = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", text or "")).strip()
    return text if len(text) <= limit else text[:limit].rstrip() + "…"


def safe_err(e):
    """錯誤訊息遮蔽金鑰"""
    msg = f"{type(e).__name__}: {e}"
    for k in (FINNHUB_API_KEY, FRED_API_KEY):
        if k:
            msg = msg.replace(k, "***")
    return clean(msg, 120)


def attempt(fn, *args, **kwargs):
    """回傳 (結果, 錯誤訊息)：平行抓取時一項失敗不影響其他項"""
    try:
        return fn(*args, **kwargs), None
    except Exception as e:
        return None, safe_err(e)


def gather(tasks, workers, deadline=8):
    """平行執行 tasks = {名稱: (連線池, 無參數函式)}，workers = {連線池: 同時請求數}。
    超過 deadline 秒還沒回來的重送一次（Yahoo 偶爾有請求卡 30 秒才逾時，整格不必跟著等）。
    回傳 {名稱: (結果, 錯誤訊息)}"""
    pools = {p: ThreadPoolExecutor(n) for p, n in workers.items()}
    jobs = {k: pools[p].submit(attempt, fn) for k, (p, fn) in tasks.items()}
    wait(jobs.values(), timeout=deadline)
    for k, f in jobs.items():
        if not f.done():
            p, fn = tasks[k]
            jobs[k] = pools[p].submit(attempt, fn)
    wait(jobs.values(), timeout=deadline)
    for ex in pools.values():
        ex.shutdown(wait=False, cancel_futures=True)     # 卡住的請求留在背景自行逾時
    return {k: f.result() if f.done() else (None, f"逾時（超過 {2 * deadline} 秒）") for k, f in jobs.items()}


FINNHUB_CACHE = {}   # 5 分鐘內重跑同一格直接用上次結果（免費版每分鐘只有 60 次，重跑很容易超過）


def finnhub(path, **params):
    if not FINNHUB_API_KEY:
        raise ValueError("未設定 FINNHUB_API_KEY")
    key = (path, tuple(sorted(params.items())))
    if key in FINNHUB_CACHE and time.time() - FINNHUB_CACHE[key][0] < 300:
        return FINNHUB_CACHE[key][1]
    for i in range(2):
        r = HTTP.get(f"https://finnhub.io/api/v1/{path}", params={**params, "token": FINNHUB_API_KEY},
                     timeout=TIMEOUT)
        if r.status_code != 429 or i:
            break
        time.sleep(1.5)                     # 超過每秒上限：稍等再試一次
    r.raise_for_status()
    data = r.json()
    if isinstance(data, dict) and data.get("error"):
        raise ValueError(data["error"])
    FINNHUB_CACHE[key] = (time.time(), data)
    return data

Finnhub： 已設定 ｜FRED： 已設定


## 2. Prices

In [2]:
try:
    from yfinance.data import YfData    # yfinance 自己的 Yahoo 連線（處理 cookie / crumb）
except ImportError:                     # yfinance 改版找不到時，closes() 全部改用 yf.download
    YfData = None

SPARK = "https://query1.finance.yahoo.com/v7/finance/spark"   # Yahoo 批次日線，一次最多 20 檔


def trade_dates(stamps):
    """Yahoo 日 K 時間戳 → 美東交易日。
    期貨（黃金、原油、美元指數）的日 K 標在美東 00:00；美東 18:00 後另有一根即時 K 棒，屬於下一個交易日，
    不能當成今天的收盤（yf.download 會把它併進今天）。"""
    t = [datetime.fromtimestamp(x, NY) for x in stamps]
    days = [x.date() for x in t]
    if len(t) >= 2 and t[-1].time() != t[-2].time() and t[-1].hour >= 18:
        days[-1] += timedelta(days=1)
    return days


def spark(batch):
    """{代號: (近 10 日收盤序列, 第一筆之前的收盤)}"""
    r = YfData().get(SPARK, params={"symbols": ",".join(batch), "range": "10d", "interval": "1d"}, timeout=TIMEOUT)
    r.raise_for_status()
    out = {}
    for item in r.json()["spark"]["result"] or []:
        resp = (item.get("response") or [{}])[0]
        stamps = resp.get("timestamp") or []
        close = (resp.get("indicators", {}).get("quote") or [{}])[0].get("close") or []
        if stamps and len(stamps) == len(close):
            s = pd.Series(close, index=pd.to_datetime(trade_dates(stamps)), dtype=float).dropna()
            out[item["symbol"]] = (s[~s.index.duplicated(keep="last")], resp["meta"].get("chartPreviousClose"))
    return out


def closes(symbols):
    """近 10 個交易日收盤價（列：美東交易日，欄：代號）。
    先用 Yahoo 批次端點（20 檔一組、多組同時抓，500 檔約 2–3 秒）；抓不到的再用 yf.download 補（較慢）。"""
    symbols = list(dict.fromkeys(symbols))
    data = {}
    if YfData is not None:
        batches = [symbols[i:i + 20] for i in range(0, len(symbols), 20)]
        res = gather({i: ("yahoo", partial(spark, b)) for i, b in enumerate(batches)}, {"yahoo": YAHOO_WORKERS})
        for part, _ in res.values():
            data.update(part or {})
    wanted = set(symbols)
    c = pd.DataFrame({s: v[0] for s, v in data.items() if s in wanted and not v[0].empty})
    missing = [s for s in symbols if s not in c]
    if missing:
        df = yf.download(missing, period="10d", interval="1d", auto_adjust=False, progress=False)
        if not df.empty and "Close" in df:
            extra = df["Close"]
            extra = extra.to_frame(missing[0]) if isinstance(extra, pd.Series) else extra
            c = c.join(extra.dropna(axis=1, how="all"), how="outer") if not c.empty else extra
    if c.empty:
        raise RuntimeError("Yahoo Finance 沒有回傳資料（請檢查網路）")
    c = c.sort_index()
    c.attrs["prev_close"] = {s: v[1] for s, v in data.items()}
    return c


def last_two(c, as_of):
    """{代號: (收盤, 前一日收盤)}，以美股收盤日為準（期貨可能已有隔日資料）。
    Yahoo 只給最新一筆日 K 的代號（S&P 500 類股指數），用該筆之前的收盤當前一日收盤。"""
    prev0 = c.attrs.get("prev_close", {})
    out = {}
    for s, col in c.items():
        col = col.dropna()
        v = col.loc[:as_of]
        if len(v) >= 2:
            out[s] = (float(v.iloc[-1]), float(v.iloc[-2]))
        elif len(col) == 1 and col.index[0] == as_of and prev0.get(s):
            out[s] = (float(v.iloc[-1]), float(prev0[s]))
    return out


def chg(s, q=None):
    last, prev = (q or Q)[s]
    return last - prev, (last / prev - 1) * 100, last


symbols = [*INDICES, *ADRS, *FOCUS, *ASSETS, *BREADTH] + [x for _, idx, etf in SECTORS.values() for x in (idx, etf) if x]
C = closes(symbols)
AS_OF = C[[s for s in INDICES if s in C]].dropna(how="all").index[-1]
Q = last_two(C, AS_OF)
# 類股漲跌：優先用 S&P 500 類股指數，沒有資料才用 SPDR ETF
SECTOR_PCT, via_etf, no_sector = {}, [], []
for en, (zh, idx, etf) in SECTORS.items():
    s = idx if idx in Q else etf if etf in Q else None
    if s:
        SECTOR_PCT[en] = chg(s)[1]
        if s == etf:
            via_etf.append(f"{zh}（{etf}）")
    else:
        no_sector.append(zh)
RANKED = sorted(SECTOR_PCT, key=SECTOR_PCT.get, reverse=True)
missing = [s for s in [*INDICES, *ADRS, *FOCUS, *ASSETS, *BREADTH] if s not in Q] + no_sector
print(f"資料日期：{AS_OF:%Y-%m-%d}（美東收盤）")
print("類股改用 ETF：" + "、".join(via_etf) if via_etf else "類股全部使用 S&P 500 類股指數")
print("缺資料：" + "、".join(missing) if missing else "全部抓取成功")

資料日期：2026-11-10（美東收盤）
類股全部使用 S&P 500 類股指數
全部抓取成功


## 3. Report Data Blocks

In [3]:
def updown(d, digits=2):
    return f"{'漲' if d >= 0 else '跌'}{abs(d):.{digits}f}"


out = ["《股市》", ""]
for s, name in {**INDICES, **ADRS}.items():
    if s not in Q:
        out += [f"🔘{name} n/a", ""]
        continue
    d, p, last = chg(s)
    if s == "^VIX":
        out += [f"🔘{name} {updown(d, 1)} ({p:+.2f}%) 收 {last:.2f}", ""]
    elif s in INDICES:
        out += [f"🔘{name} {updown(d)} ({p:+.2f}%) 收 {last:.0f}", ""]
    else:
        out += [f"🔘{name} {updown(d)} ({p:+.2f}%) 收 {last:.2f}", ""]
out += ["（盤勢評論：盤前 → 盤中 → 收盤，2–4 段）", "", "（類股評論：最強類股 → 次強 → 最弱，各一段）", ""]
out += ["🔹強勢類股：", ""] + [x for s in RANKED[:3] for x in (f"{SECTORS[s][0]} {SECTOR_PCT[s]:+.2f}%", "")]
out += ["🔹弱勢類股：", ""] + [x for s in RANKED[::-1][:3] for x in (f"{SECTORS[s][0]} {SECTOR_PCT[s]:+.2f}%", "")]
focus = sorted((s for s in FOCUS if s in Q), key=lambda s: chg(s)[1], reverse=True)
out += ["🔹焦點股： " + "、".join(f"{FOCUS[s]} {chg(s)[1]:+.2f}%" for s in focus), ""]
out += ["《美債/美元/黃金/原油》", ""]
for s, name in ASSETS.items():
    if s not in Q:
        out += [f"🔘{name} n/a", ""]
    elif s == "^TNX":
        last, prev = Q[s]
        out += [f"🔘{name} {round((last - prev) * 100):+d}bps 收{last:.2f}%", ""]
    else:
        d, p, last = chg(s)
        out += [f"🔘{name} {p:+.2f}% 收 {last:.2f}", ""]
out += [".", ""] + [f"🔹{k}：（待撰寫）\n" for k in ("美債", "美元", "黃金", "原油")]
out += ["《重點訊息》", "", "🔹利多：", "", "1. （待撰寫）", "", "🔹利空：", "", "1. （待撰寫）"]
print("\n".join(out))

《股市》

🔘道瓊 跌157.86 (-0.30%) 收 52232

🔘標普 跌29.01 (-0.38%) 收 7521

🔘納指 跌187.26 (-0.74%) 收 25066

🔘費半 跌35.60 (-0.35%) 收 9998

🔘VIX 跌0.5 (-2.31%) 收 19.46

🔘台積 漲7.86 (+2.02%) 收 396.48

🔘聯電 漲0.15 (+1.01%) 收 15.04

🔘日月光 漲1.12 (+3.26%) 收 35.48

（盤勢評論：盤前 → 盤中 → 收盤，2–4 段）

（類股評論：最強類股 → 次強 → 最弱，各一段）

🔹強勢類股：

金融 +0.49%

非必需消費 +0.46%

能源 +0.32%

🔹弱勢類股：

資訊科技 -1.40%

原材料 -0.55%

公用事業 -0.23%

🔹焦點股： Amazon +1.48%、AMD +1.30%、Tesla +0.46%、Goldman Sachs +0.44%、Morgan Stanley +0.25%、Meta +0.00%、JP Morgan -0.00%、Google -0.27%、Apple -0.62%、Microsoft -2.20%、SuperMicro -2.83%、NVIDIA -5.47%

《美債/美元/黃金/原油》

🔘美債 +12bps 收4.98%

🔘美元 +0.57% 收 102.23

🔘黃金 -2.05% 收 4030.20

🔘西德州 -0.93% 收 86.04

🔘布蘭特 -1.17% 收 91.49

.

🔹美債：（待撰寫）

🔹美元：（待撰寫）

🔹黃金：（待撰寫）

🔹原油：（待撰寫）

《重點訊息》

🔹利多：

1. （待撰寫）

🔹利空：

1. （待撰寫）


## 4. Sector detail, top movers, market breadth

In [4]:
def sp500():
    """S&P 500 成分股（維基百科），快取 7 天"""
    if os.path.exists(SP500_CACHE) and time.time() - os.path.getmtime(SP500_CACHE) < 7 * 86400:
        return pd.read_csv(SP500_CACHE)
    r = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers=UA, timeout=TIMEOUT)
    r.raise_for_status()
    t = next(t for t in pd.read_html(io.StringIO(r.text)) if {"Symbol", "GICS Sector"} <= set(t.columns))
    df = t.rename(columns={"Symbol": "symbol", "Security": "name", "GICS Sector": "sector",
                           "GICS Sub-Industry": "industry"})[["symbol", "name", "sector", "industry"]]
    df["symbol"] = df["symbol"].astype(str).str.replace(".", "-", regex=False)   # BRK.B → BRK-B
    df.to_csv(SP500_CACHE, index=False)
    return df


def tag(r):
    return f"{r['name']}（{r['symbol']} {r['chg']:+.2f}%）"


UNI = sp500()
QU = last_two(closes(UNI["symbol"]), AS_OF)
UNI["chg"] = [chg(s, QU)[1] if s in QU else None for s in UNI["symbol"]]
UNI = UNI.dropna(subset=["chg"])

adv, dec = int((UNI["chg"] > 0).sum()), int((UNI["chg"] < 0).sum())
lines = [f"**市場廣度**：上漲 {adv} 家、下跌 {dec} 家（漲跌家數比 {adv / max(dec, 1):.2f}）"]
if all(s in Q for s in BREADTH):
    lines[0] += f"｜等權重 RSP {chg('RSP')[1]:+.2f}% vs 市值加權 SPY {chg('SPY')[1]:+.2f}%"
MOVERS = []
for en in RANKED[:3] + RANKED[::-1][:3]:
    sub = UNI[UNI["sector"] == en].sort_values("chg", ascending=False)
    if sub.empty:
        continue
    ind = sub.groupby("industry")["chg"].agg(["mean", "count"]).sort_values("mean", ascending=False)
    fmt_ind = lambda rows: "、".join(f"{i} {m:+.2f}%（{int(n)}檔）" for i, (m, n) in rows.iterrows()) or "—"
    top = sub[sub["chg"] > 0].head(MOVERS_PER_SECTOR)
    bottom = sub[sub["chg"] < 0].tail(MOVERS_PER_SECTOR).iloc[::-1]
    for _, r in pd.concat([top, bottom]).iterrows():
        if r["symbol"] not in {m["symbol"] for m in MOVERS}:
            MOVERS.append(r)
    lines += ["", f"### {SECTORS[en][0]} {SECTOR_PCT[en]:+.2f}%",
              f"- 子產業最強：{fmt_ind(ind[ind['mean'] > 0].head(3))}",
              f"- 子產業最弱：{fmt_ind(ind[ind['mean'] < 0].tail(3).iloc[::-1])}",
              f"- 領漲：{'、'.join(tag(r) for _, r in top.iterrows()) or '—'}",
              f"- 領跌：{'、'.join(tag(r) for _, r in bottom.iterrows()) or '—'}"]
display(Markdown("\n".join(lines)))

**市場廣度**：上漲 247 家、下跌 250 家（漲跌家數比 0.99）｜等權重 RSP -0.03% vs 市值加權 SPY -0.37%

### 金融 +0.49%
- 子產業最強：Reinsurance +2.32%（1檔）、Multi-line Insurance +1.95%（3檔）、Property & Casualty Insurance +1.75%（8檔）
- 子產業最弱：Consumer Finance -1.24%（3檔）、Regional Banks -0.55%（6檔）、Investment Banking & Brokerage -0.20%（6檔）
- 領漲：Global Payments（GPN +3.91%）、Cboe Global Markets（CBOE +3.41%）、American International Group（AIG +3.38%）
- 領跌：Block, Inc.（XYZ -3.36%）、Coinbase（COIN -2.92%）、Robinhood Markets（HOOD -2.39%）

### 非必需消費 +0.46%
- 子產業最強：Broadline Retail +1.79%（2檔）、Casinos & Gaming +0.91%（3檔）、Leisure Products +0.74%（1檔）
- 子產業最弱：Homebuilding -4.46%（4檔）、Computer & Electronics Retail -2.05%（1檔）、Automotive Retail -1.91%（3檔）
- 領漲：Marriott International（MAR +2.66%）、eBay Inc.（EBAY +2.11%）、Hilton Worldwide（HLT +1.75%）
- 領跌：Lennar（LEN -5.37%）、NVR, Inc.（NVR -4.76%）、Carvana（CVNA -4.25%）

### 能源 +0.32%
- 子產業最強：Oil & Gas Storage & Transportation +1.99%（4檔）、Oil & Gas Exploration & Production +1.53%（9檔）、Integrated Oil & Gas +0.03%（2檔）
- 子產業最弱：Oil & Gas Refining & Marketing -0.90%（3檔）、Oil & Gas Equipment & Services -0.81%（3檔）
- 領漲：EQT Corporation（EQT +4.19%）、Targa Resources（TRGP +2.81%）、Devon Energy（DVN +2.38%）
- 領跌：Valero Energy（VLO -1.69%）、Baker Hughes（BKR -1.40%）、Texas Pacific Land Corporation（TPL -1.31%）

### 資訊科技 -1.40%
- 子產業最強：Technology Distributors +2.89%（1檔）、Internet Services & Infrastructure +2.42%（3檔）、IT Consulting & Other Services +0.52%（4檔）
- 子產業最弱：Electronic Components -2.97%（3檔）、Electronic Manufacturing Services -1.40%（3檔）、Technology Hardware, Storage & Peripherals -1.04%（10檔）
- 領漲：Akamai Technologies（AKAM +4.26%）、Broadcom（AVGO +3.91%）、Twilio（TWLO +3.66%）
- 領跌：First Solar（FSLR -5.55%）、Nvidia（NVDA -5.47%）、Coherent Corp.（COHR -4.68%）

### 原材料 -0.55%
- 子產業最強：Steel +3.33%（2檔）、Fertilizers & Agricultural Chemicals +1.77%（2檔）、Commodity Chemicals +1.68%（1檔）
- 子產業最弱：Gold -4.15%（1檔）、Copper -3.24%（1檔）、Specialty Chemicals -0.92%（6檔）
- 領漲：Steel Dynamics（STLD +3.76%）、Nucor（NUE +2.90%）、CF Industries（CF +2.58%）
- 領跌：Albemarle Corporation（ALB -4.23%）、Newmont（NEM -4.15%）、Freeport-McMoRan（FCX -3.24%）

### 公用事業 -0.23%
- 子產業最強：Gas Utilities +1.17%（1檔）、Water Utilities +0.76%（1檔）、Multi-Utilities +0.26%（12檔）
- 子產業最弱：Electric Utilities -0.36%（15檔）、Independent Power Producers & Energy Traders -0.08%（2檔）
- 領漲：Atmos Energy（ATO +1.17%）、CenterPoint Energy（CNP +1.05%）、American Water Works（AWK +0.76%）
- 領跌：Constellation Energy（CEG -3.02%）、Vistra Corp.（VST -2.05%）、NextEra Energy（NEE -0.68%）

## 5. Why stocks moved: analyst ratings + headlines

In [5]:
ACTION = {"up": "升評", "down": "降評", "init": "首評", "main": "維持", "reit": "重申"}
DAY = AS_OF.date()
HOUR = {"bmo": "盤前", "amc": "盤後", "dmh": "盤中"}


def analyst(sym, days=3):
    """Yahoo 分析師評等變動（近 N 天）"""
    ud = yf.Ticker(sym).upgrades_downgrades
    if ud is None or ud.empty:
        return []
    idx = pd.to_datetime(ud.index)
    idx = idx.tz_localize(None) if idx.tz is not None else idx
    ud = ud[idx >= pd.Timestamp(AS_OF) - pd.Timedelta(days=days)]
    out = []
    for _, r in ud.iterrows():
        text = f"{r.get('Firm', '')} {ACTION.get(r.get('Action'), r.get('Action', ''))}"
        if r.get("FromGrade") and r.get("FromGrade") != r.get("ToGrade"):
            text += f"：{r['FromGrade']} → {r.get('ToGrade', '')}"
        elif r.get("ToGrade"):
            text += f"：{r['ToGrade']}"
        if pd.notna(r.get("currentPriceTarget")) and r.get("currentPriceTarget"):
            prior = r.get("priorPriceTarget")
            text += f"，目標價 {prior:g} → {r['currentPriceTarget']:g}" if pd.notna(prior) and prior else \
                f"，目標價 {r['currentPriceTarget']:g}"
        out.append(text)
    return out


def stock_news(sym, n=3):
    """有 Finnhub 金鑰用 Finnhub（含摘要），否則用 Yahoo"""
    if FINNHUB_API_KEY:
        data = finnhub("company-news", symbol=sym, **{"from": f"{DAY - timedelta(days=1):%Y-%m-%d}",
                                                       "to": f"{DAY + timedelta(days=1):%Y-%m-%d}"})
        data = sorted((d for d in data if d.get("headline")), key=lambda d: d.get("datetime", 0), reverse=True)[:n]
        return [f"[{clean(d['headline'], 200)}]({d.get('url', '')}) — {d.get('source', '')}"
                + (f"：{clean(d['summary'], 160)}" if d.get("summary") else "") for d in data]
    out = []
    for it in (yf.Ticker(sym).news or [])[:n]:
        c = it.get("content", it)            # yfinance 新舊兩種格式
        url = (c.get("canonicalUrl") or {}).get("url") or c.get("link", "")
        src = (c.get("provider") or {}).get("displayName") or c.get("publisher", "")
        if c.get("title"):
            out.append(f"[{c['title']}]({url}) — {src}")
    return out


# ---- 當日財報（Finnhub）：寫「重點訊息」與「盤後」段落用 ----
def beat(actual, est):
    if actual is None or est is None:
        return ""
    return "（優於預期）" if actual > est else "（不如預期）" if actual < est else "（符合預期）"


def earnings_table(data):
    sp = UNI.set_index("symbol")
    cal = sorted((d for d in data["earningsCalendar"] if d.get("symbol") in sp.index),
                 key=lambda d: -(d.get("revenueEstimate") or 0))
    if not cal:
        return "- 當日無 S&P 500 成分股公布財報"
    rev = lambda v: f"{v / 1e8:,.1f}" if v else "—"
    eps = lambda v: f"{v:.2f}" if v is not None else "—"
    rows = ["| 公司 | 時段 | EPS 實際 / 預估 | 營收 實際 / 預估（億美元） | 當日股價 |", "|:---|:---|---:|---:|---:|"]
    for d in cal[:20]:
        px = f"{sp.loc[d['symbol'], 'chg']:+.2f}%" if pd.notna(sp.loc[d["symbol"], "chg"]) else "—"
        rows.append(f"| {sp.loc[d['symbol'], 'name']}（{d['symbol']}） | {HOUR.get(d.get('hour'), d.get('hour') or '—')} | "
                    f"{eps(d.get('epsActual'))} / {eps(d.get('epsEstimate'))}{beat(d.get('epsActual'), d.get('epsEstimate'))} | "
                    f"{rev(d.get('revenueActual'))} / {rev(d.get('revenueEstimate'))} | {px} |")
    return "\n".join(rows) + "\n\n盤後公布的財報，當日股價尚未反映，實際數字也可能稍晚才更新。"


# 評等（Yahoo）、新聞與財報（Finnhub）分兩組同時抓
tasks = {"cal": ("finnhub", partial(finnhub, "calendar/earnings", **{"from": f"{DAY:%Y-%m-%d}", "to": f"{DAY:%Y-%m-%d}"}))}
for r in MOVERS:
    tasks["analyst", r["symbol"]] = ("yahoo", partial(analyst, r["symbol"]))
    tasks["news", r["symbol"]] = ("finnhub", partial(stock_news, r["symbol"]))
res = gather(tasks, {"yahoo": YAHOO_WORKERS,
                     "finnhub": FINNHUB_WORKERS if FINNHUB_API_KEY else 2})   # 沒金鑰時新聞也走 Yahoo，壓低同時請求數

lines, analyst_errs = [], []
for r in MOVERS:
    (ratings, ratings_err), (news, news_err) = res["analyst", r["symbol"]], res["news", r["symbol"]]
    lines.append(f"**{tag(r)}**")
    lines += [f"- 📊 {a}" for a in ratings or []]
    lines += [f"- 📰 {n}" for n in news or []]
    if news_err:
        lines.append(f"- 📰 （新聞無法取得：{news_err}）")
    if ratings_err:
        analyst_errs.append(ratings_err)
    if not ratings and not news and not news_err:
        lines.append("- （近期無評等變動或新聞）")
    lines.append("")
if analyst_errs:
    lines.append(f"⚠️ 分析師評等 {len(analyst_errs)}/{len(MOVERS)} 檔無法取得（Yahoo 限流時常見，稍後重跑本格即可）：{analyst_errs[0]}")
display(Markdown("\n".join(lines)))

cal, err = res["cal"]
body, err = (None, err) if err else attempt(earnings_table, cal)
display(Markdown(f"**當日財報｜{DAY:%Y-%m-%d}（S&P 500 成分股，依營收規模排序）**\n\n{body or f'- 無法取得：{err}'}"))

**Global Payments（GPN +3.91%）**
- 📊 Mizuho 維持：Outperform，目標價 110 → 115
- 📊 Barclays 維持：Equal-Weight，目標價 89 → 97
- 📊 Cantor Fitzgerald 維持：Neutral，目標價 91 → 103
- 📊 JP Morgan 維持：Neutral，目標價 95 → 100
- 📰 [Michael Burry makes a bold call on struggling payments giant](https://finnhub.io/api/news?id=998bfcbb250843b987a3aedb6d1a4679a550b5d9ea07c371bfcf34d250de8859) — Yahoo：Wall Street cut its ratings, so he doubled down and raised his target.
- 📰 [S&P 500 Top Gainers and Losers for Today's Session](https://finnhub.io/api/news?id=6fce76dd7876e030d37b388ebacf02dc819d615b61dee883f327a61f69708576) — ChartMill：S&P 500 market movers for today's session: Charter Communications led gainers at 9.68%, while Carvana led losers at 6.32%.
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=7298d9b65a4f1c1d8597bf187062a57b8ac62c13c137ce3889144a6764273edf) — ChartMill：S&P 500 top movers for today's US session: Take-Two Interactive led gainers at 7.57%, while GE Vernova led decliners at 7.19%.

**Cboe Global Markets（CBOE +3.41%）**
- 📰 [CBOE Global (CBOE) is a Great Momentum Stock: Should You Buy?](https://finnhub.io/api/news?id=75f0f1275b597c22d1b57e8e81fdb2965e21791e70b45935f1146fef21dd88ab) — Yahoo：Does CBOE Global (CBOE) have what it takes to be a top stock pick for momentum investors? Let's find out.
- 📰 [Looking for a Growth Stock? 3 Reasons Why CBOE (CBOE) is a Solid Choice](https://finnhub.io/api/news?id=19c4f3128205236ab219fd88cf81e2ee973522009a30786d461bc2894d54cfa7) — Yahoo：CBOE (CBOE) possesses solid growth attributes, which could help it handily outperform the market.
- 📰 [Cboe Expands Prediction Markets Suite to Small-Caps with Launch of Mini-Russell 2000 Index Binary Options](https://finnhub.io/api/news?id=90a4ee7abaa010052a44ba28f24163db9c081384e87ea72816e05467f42c09e6) — Yahoo：Cboe Global Markets, Inc. (Cboe: CBOE), a leading global markets operator and pioneer in equity and index derivatives, today announced the launch of mini-Russel…

**American International Group（AIG +3.38%）**
- 📊 UBS 維持：Buy，目標價 100 → 103
- 📊 Piper Sandler 維持：Neutral，目標價 84 → 85
- 📰 [S&P 500 Top Gainers and Losers for Today's Session](https://finnhub.io/api/news?id=6fce76dd7876e030d37b388ebacf02dc819d615b61dee883f327a61f69708576) — ChartMill：S&P 500 market movers for today's session: Charter Communications led gainers at 9.68%, while Carvana led losers at 6.32%.
- 📰 [Corebridge Financial: Merger Synergies Ahead](https://finnhub.io/api/news?id=54073fd05e759c6828afb3afce112df9ea212cb9408d31ab689c05b1d91e9318) — SeekingAlpha：Corebridge Financial, Inc. is a covered-call candidate at best: its Equitable merger could generate over $5B in adjusted operating earnings. Click for more on C…
- 📰 [Can American Airlines Stock Fall 30%?](https://finnhub.io/api/news?id=64e6c114b454d79d592dbf7fe0daf1cbc175e58edacb7c924d39ae686c5ebfdb) — Yahoo：For questions about this article or underlying research, email help@trefis.com.

**Block, Inc.（XYZ -3.36%）**
- 📊 Cantor Fitzgerald 維持：Overweight，目標價 95 → 102
- 📰 [Block’s Cash App weaves payment tech into bandanas](https://finnhub.io/api/news?id=7f7bd8f8ab11dd0a3df1aac3be604254014324653f3dbe4fc3cbc2f7389d072a) — Yahoo：The peer-to-peer payment provider will soon begin selling its digital tags attached to bandanas.
- 📰 [Block, Estee Lauder Among Stocks With Rising Profit Estimates](https://finnhub.io/api/news?id=7c751b948b9fb3296641dc2ecfd373e638c719ea76500df398ad0953a47f3c6a) — Yahoo：Four stocks are featured on IBD's screen for rising profit estimates. The stocks are also in or near buy zones.
- 📰 [Affirm Printed a “No-Apologies Quarter” and Still Lost 16% in a Month. BMO Sees 53% Upside](https://finnhub.io/api/news?id=e4883ffaa5de2eb82b1b8b567c1fbc17fa2de889b28037d78ac0df9dfaf72fec) — Yahoo：Key TakeawaysAffirm shares fell 16% in the past month as a hot inflation report, a post-midterm sell-off and credit warnings from Upstart and Block hit consumer…

**Coinbase（COIN -2.92%）**
- 📊 Freedom Broker 維持：Buy，目標價 185 → 175
- 📰 [Coinbase's national trust charter approval is a major win for crypto custody](https://finnhub.io/api/news?id=a864633047d3e984aa21ac015ecb26cc1f508837ead32443f1c70e4e89ff1e73) — Yahoo：A top financial regulatory cop has granted Coinbase — the world's biggest crypto exchange — a national trust charter.
- 📰 [7.1 Million Bitcoin Sit in Wallets a Quantum Computer Could One Day Crack. Is Yours One of Them?](https://finnhub.io/api/news?id=3b28b2b7b93d6a729c5d41db2baed07cfa0e09abd5d57d71bb0234a51740bea6) — Yahoo：A third of all Bitcoin has a public key sitting in plain view, yet the real pressure on price is coming from somewhere else entirely. Find out which coins count…
- 📰 [Coinbase Gets National Bank Charter Approval](https://finnhub.io/api/news?id=98ac379601e0897c5484faee8bb6fc7939b0c910ea1716e8475269ed14f397b9) — Yahoo：Bloomberg's Olga Kharif joins Isabelle Lee and Tim Stenovec on "Bloomberg Crypto." Coinbase, the largest US crypto exchange, said a key regulator has fully sign…

**Robinhood Markets（HOOD -2.39%）**
- 📰 [5 Best AI-Powered Financial Stocks to Buy According to Analysts](https://finnhub.io/api/news?id=b8b84a3e460a4d96348613bcf239fee3dd5f492c60bc2d309b3d59b810192aaf) — Yahoo：Banks and lenders have spent years promising that artificial intelligence will cut costs and speed up decisions. This autumn, the people who supervise them star…
- 📰 [Securitize Stock Jumps 24%, Defying Crypto Slump. Its Tokenized Assets Hit $5 Billion.](https://finnhub.io/api/news?id=a7850f2fc60ac93f7e76ee6e553187ceea0d117a754b76635f1a778baef5f2c7) — Yahoo：Securitize stock is heading for its best single-day percentage gain on record even after reporting mixed earnings.
- 📰 [Coinbase Falls 4%, Robinhood Slides 3% as Hot Inflation Data Drags Bitcoin Lower; Webull Declines 2%](https://finnhub.io/api/news?id=5ec16cb199e7e03b22c606d4b0a77d8c579f7f692dbf4298d01c80e476e89545) — Yahoo：Crypto-exposed trading platforms are slipping as an inflation report that came in above forecasts sends Bitcoin (CRYPTO:BTC) lower and revives expectations for…

**Marriott International（MAR +2.66%）**
- 📰 [Marriott’s Dividend Yields 0.7%. Its $4.8 Billion Cash Return Plan Changes the Math](https://finnhub.io/api/news?id=3784445c84f64a1263323178ea50d5661162a9baf8545edb1e649cd9655e85ed) — Yahoo：A sub-1% yield makes Marriott look like a poor pick for income investors. The real payout hides in a place most dividend screens never check, and one quirk on i…
- 📰 [Stock Market Today: Dow Dips On Hot Inflation Data; Nvidia Falls Again, Anthropic Sets IPO Range (Live Coverage)](https://finnhub.io/api/news?id=da4b6468f7b07843e1b9de42fc190efda0a049a2abd6c729c74866c400d1cd24) — Yahoo：Stock Market Today: The Dow Jones index falls Tuesday on a hot inflation report. Anthropic prices its IPO range. Nvidia sells off.
- 📰 [Vacarya Surpasses 500 Properties, Becoming One of North America's Largest Rental Arbitrage Companies](https://finnhub.io/api/news?id=c85531b9cc839c4ad0ec9d3b68b92b3ced0934cea5f998c34ae1a2d6ae24bc3a) — Yahoo：The Toronto-founded hospitality brand now operates in more than 30 cities across the U. and Canada with a network of more than 110 independent partners.

**eBay Inc.（EBAY +2.11%）**
- 📰 [Did The Market Read eBay Stock Right?](https://finnhub.io/api/news?id=2e603b6d12903fd25379e9f30ff8249a7ad14cd46fe04654885d0e76ca62694c) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [ClearJet Adds Veteran Operator Brent Bellm to Board, Bringing Scale-Up Experience Across Payments, Travel and Enterprise E-commerce](https://finnhub.io/api/news?id=a2156fb1bec1a5839d4b4913245db8662873b3cff6f83559e075c71606f1ecb0) — Yahoo：ClearJet, the parcel logistics company behind the SuperCarrier network, today announced that Brent Bellm has joined its Board of Directors as an independent dir…
- 📰 [Multilogin Data Shows How Cloud Phones Are Used Beyond Social Media](https://finnhub.io/api/news?id=caa01e1afac5af1f8f1664ab8b4e62ddc9292d1a965a07f66c5925c98fae3486) — Yahoo：Explore how Multilogin cloud phones fit different needs, from social media and messaging apps to online marketplaces and AI tools. How People Use Multilogin Clo…

**Hilton Worldwide（HLT +1.75%）**
- 📰 [Marriott’s Dividend Yields 0.7%. Its $4.8 Billion Cash Return Plan Changes the Math](https://finnhub.io/api/news?id=3784445c84f64a1263323178ea50d5661162a9baf8545edb1e649cd9655e85ed) — Yahoo：A sub-1% yield makes Marriott look like a poor pick for income investors. The real payout hides in a place most dividend screens never check, and one quirk on i…
- 📰 [NHAAN Resort & Spa Hoi An, Tapestry Collection by Hilton Introduces Plait, Its Signature Riverside Restaurant](https://finnhub.io/api/news?id=dcb53899122ab9ca3a01be92f8c3f052fbd81d2da8cb1ad7f59f25b2c95f67c2) — Yahoo：NHAAN Resort & Spa Hoi An, Tapestry Collection by Hilton, the first Tapestry Collection by Hilton resort in Vietnam, has unveiled Plait, its signature riverside…

**Lennar（LEN -5.37%）**
- 📰 [S&P 500 Top Gainers and Losers for Today's Session](https://finnhub.io/api/news?id=6fce76dd7876e030d37b388ebacf02dc819d615b61dee883f327a61f69708576) — ChartMill：S&P 500 market movers for today's session: Charter Communications led gainers at 9.68%, while Carvana led losers at 6.32%.
- 📰 [S&P 500 Gap-Down Stocks: Nov. 10, 2026 Session](https://finnhub.io/api/news?id=4b6e7e9ee293a5892be5922c2159d3b2fd8682a7b7465976d0cf38d9b250a668) — ChartMill：S&P 500 gap-down stocks on Nov. 10, 2026, including GEV, APP, LEN, DHI, BBY, MRNA, PHM, NVDA, FSLR and ETN, with session moves and gap details.
- 📰 [Title Resources Group Earns Great Place To Work Certification™ for Third Consecutive Year](https://finnhub.io/api/news?id=c41633e2f03bf7532d1656c11f1ae55f359b00483dc6dc4c2da2b985e883c87b) — Yahoo：Title Resources Group (TRG), one of the nation's leading title insurance underwriters, is proud to be Certified™ by Great Place To Work® for the third year in a…

**NVR, Inc.（NVR -4.76%）**
- 📰 [What Invesco S&P 500 Equal Weight ETF (RSP) Bought: Moderna Leads on Nov. 6](https://finnhub.io/api/news?id=79e6c963eac81d33c75424bc85a72408ccc9b9611730e2a21e893c9b0ff3287e) — Yahoo：The fund added $291.5 million in gross purchases on Friday, increasing all 503 positions and selling nothing.
- 📰 [Premarket movers: Nvidia slides on Huang probe, NVR extends rally](https://finnhub.io/api/news?id=42a95e375ec3cd38babc61218f770fbc68b249d8ab817f6acc3132d41f80a50e) — Yahoo：Investing.com -- Stock futures rose early Monday as traders watched technology shares following last week’s volatility, awaiting key inflation data and earnings…
- 📰 [Invesco S&P 500 Equal Weight (RSP) Adds Vylor and Moderna on Nov. 5](https://finnhub.io/api/news?id=0bda31c4db5aa3e66c7213b9a1a7fb7040ca081a93516b681502909ee5b849f5) — Yahoo：The $101.82 billion fund bought $976.9 million of stock on Thursday, adding 503 positions and selling none.

**Carvana（CVNA -4.25%）**
- 📰 [Can Carvana Stock Fall 40%?](https://finnhub.io/api/news?id=1a7de17cc199d2ce8c07fce023bda6db1dbec98055a38a995e4f8bcfc495d30f) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [S&P 500 Top Gainers and Losers for Today's Session](https://finnhub.io/api/news?id=6fce76dd7876e030d37b388ebacf02dc819d615b61dee883f327a61f69708576) — ChartMill：S&P 500 market movers for today's session: Charter Communications led gainers at 9.68%, while Carvana led losers at 6.32%.
- 📰 ['Just Keep Saying No'— Former Car Finance Manager Says Simple Trick Could Save You Thousands](https://finnhub.io/api/news?id=75e1ca81f0d2237f49d11fb6b707b60e8f638971b8f980903846e36dcd57aed0) — Yahoo：Buying a car can be exhausting. You spend hours comparing prices, negotiating with salespeople and trying to get a decent interest rate. But even after you’ve a…

**EQT Corporation（EQT +4.19%）**
- 📰 [S&P 500 Top Gainers and Losers for Today's Session](https://finnhub.io/api/news?id=6fce76dd7876e030d37b388ebacf02dc819d615b61dee883f327a61f69708576) — ChartMill：S&P 500 market movers for today's session: Charter Communications led gainers at 9.68%, while Carvana led losers at 6.32%.
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=7298d9b65a4f1c1d8597bf187062a57b8ac62c13c137ce3889144a6764273edf) — ChartMill：S&P 500 top movers for today's US session: Take-Two Interactive led gainers at 7.57%, while GE Vernova led decliners at 7.19%.
- 📰 [Natural Gas Prices Gain 3.5% for the Week: What's Next?](https://finnhub.io/api/news?id=18d3b7b8478e8329ab25036fde85659ee09d3814bacf8ee4784e28c482a2633a) — Yahoo：Colder forecasts and steady LNG demand lift natural gas prices, while ample inventories cap gains. WMB, EQT and EE offer exposure as winter approaches.

**Targa Resources（TRGP +2.81%）**
- 📊 Morgan Stanley 維持：Overweight，目標價 359 → 362
- 📊 Scotiabank 維持：Sector Outperform，目標價 319 → 325
- 📊 Wells Fargo 維持：Overweight，目標價 324 → 330
- 📊 Jefferies 維持：Buy，目標價 326 → 311
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=7298d9b65a4f1c1d8597bf187062a57b8ac62c13c137ce3889144a6764273edf) — ChartMill：S&P 500 top movers for today's US session: Take-Two Interactive led gainers at 7.57%, while GE Vernova led decliners at 7.19%.
- 📰 [Here’s How Much You Would Have Made Owning Targa Resources Stock In The Last 10 Years](https://finnhub.io/api/news?id=f1f3da7687a49a486aa6b72f2135c1f8b43a091a17f42048951fdd285b9a814d) — Benzinga：Targa Resources (NYSE:TRGP) has outperformed the market over the past 10 years by 5.1% on an annualized basis producing an average annual return of 18.12%. Curr…
- 📰 [Wells Fargo Maintains Overweight on Targa Resources, Raises Price Target to $330](https://finnhub.io/api/news?id=1ffd803264cc452c304c65b3847ea77d58480743d4b5ba1ccca9c4e85f11c4df) — Benzinga：Wells Fargo analyst Praneeth Satish maintains Targa Resources (NYSE:TRGP) with a Overweight and raises the price target from $324 to $330.

**Devon Energy（DVN +2.38%）**
- 📊 UBS 維持：Buy，目標價 65 → 62
- 📊 JP Morgan 維持：Overweight，目標價 65 → 68
- 📊 Morgan Stanley 維持：Overweight，目標價 65 → 64
- 📊 Truist Securities 維持：Buy，目標價 61 → 62
- 📊 Susquehanna 維持：Positive，目標價 63 → 60
- 📰 [Devon Energy Q3 Earnings Call Highlights Permian Focus and Buybacks](https://finnhub.io/api/news?id=c9cd72b652d39fe33b99851a79cc36568ff96acc46896f2f8a3a06667c49602d) — Yahoo：DVN's Q3 2026 earnings call highlights Permian expansion, higher merger synergies, accelerated buybacks and a disciplined 2027 capital spending plan.
- 📰 [Is Crescent Energy's Royalty Business Its Next Big Growth Engine?](https://finnhub.io/api/news?id=7396bca749d2d16c3dcb91196f4e712c0c13f9608d85a0c097619ee9821b138e) — Yahoo：CRGY's capital-light royalties business offers cash flow without drilling costs, while the Devon deal could bring a major boost in scale and output.
- 📰 [Devon Energy (DVN) Loses 13.4% in 4 Weeks, Here's Why a Trend Reversal May be Around the Corner](https://finnhub.io/api/news?id=78fd53a9ed6378606a3349c7f5c67e7a6fc4a6b76b11d6d43c1ecfb466df68f8) — Yahoo：The heavy selling pressure might have exhausted for Devon Energy (DVN) as it is technically in oversold territory now. In addition to this technical measure, st…

**Valero Energy（VLO -1.69%）**
- 📰 [How Investors May Respond To Valero Energy (VLO) Dividend Affirmation](https://finnhub.io/api/news?id=056dfc927105d0f7fa47a934f69bfa5db62b33f8729d29d92a8ed63b85ee78c7) — Yahoo：Valero Energy's board affirmed a regular quarterly dividend of $1.20 per share, payable on December 21, 2026, to investors on record as of November 24, 2026. Th…
- 📰 [Rising Q3 transportation costs squeeze manufacturers, reward railroads and reshape trade](https://finnhub.io/api/news?id=26bfac891cbb4dfbfa3502c96a5726839727ad4e4e82ec7f9bde602bc4199a73) — Yahoo：Higher fuel costs and freight rates are pressuring supply chains, but produce varied consequences. The post Rising Q3 transportation costs squeeze manufacturers…
- 📰 [Par Pacific Is Now Undervalued: Is It a Strong Entry Point?](https://finnhub.io/api/news?id=71b207d2cd338a1a9cf5ba797eeb32e7626702bc943f7e5f36030d73e26aab74) — Yahoo：PARR trades at a steep industry discount, while flexible crude sourcing and favorable refining trends strengthen its outlook. Is now the time to buy?

**Baker Hughes（BKR -1.40%）**
- 📰 [Top 5 Activist Target Stocks for Spin-offs or Breakups](https://finnhub.io/api/news?id=cf1936646b69b639d997300457ab5ca579481df496497721a96a961edffc9e72) — Yahoo：We just published the Top 10 Activist Target Stocks for Spin-offs or Breakups. You can find the top 5 stocks below. Activist investors have increasingly targete…
- 📰 [Baker Hughes Stock Outlook: Is Wall Street Bullish or Bearish?](https://finnhub.io/api/news?id=7bad45466d6bc428cb2c40a16ed86861840ee62d8c87b7f92d2d7c1d716d89cd) — Yahoo：Baker Hughes has underperformed the broader market over the past year, but analysts are highly optimistic about the stock’s prospects.
- 📰 [PuppyGraph Raises $10M Series A to Help Enterprises Build AI Agents That Query 3,000 Tables in Subseconds](https://finnhub.io/api/news?id=16671f24aadcae0879a3fc161d2bf8269fcf2c64d30bedd36423f5ca78e844ff) — Yahoo：SANTA CLARA, Calif., November 10, 2026--PuppyGraph, the graph query engine for enterprise AI agents, today announced a $10 million Series A led by Particle Futu…

**Texas Pacific Land Corporation（TPL -1.31%）**
- 📰 [Is Crescent Energy's Royalty Business Its Next Big Growth Engine?](https://finnhub.io/api/news?id=7396bca749d2d16c3dcb91196f4e712c0c13f9608d85a0c097619ee9821b138e) — Yahoo：CRGY's capital-light royalties business offers cash flow without drilling costs, while the Devon deal could bring a major boost in scale and output.
- 📰 [10 Dividend Growth Stocks: November 2026](https://finnhub.io/api/news?id=c67b9a47ea6c615bf0d5bc6fdcfed3bbde147a7a6f6be7a86125a9b7340981eb) — SeekingAlpha：Analyst sees TPL as the only top-10 dividend growth pick for a new position; all 10 returned 12%+ over five years and trade at or below fair value. See more her…
- 📰 [Texas Pacific Land Corporation 2026 Q3 - Results - Earnings Call Presentation](https://finnhub.io/api/news?id=f7f6f550fa77f5f8fa6a49daff1cffd7ac6504ca2ee8a1afedc52a5a4409de39) — SeekingAlpha：2026-11-10. The following slide deck was published by Texas Pacific Land Corporation in conjunction with their 2026 Q3 earnings call.

**Akamai Technologies（AKAM +4.26%）**
- 📊 Baird 維持：Neutral，目標價 140 → 92
- 📰 [S&P 500 Top Gainers and Losers for Today's Session](https://finnhub.io/api/news?id=6fce76dd7876e030d37b388ebacf02dc819d615b61dee883f327a61f69708576) — ChartMill：S&P 500 market movers for today's session: Charter Communications led gainers at 9.68%, while Carvana led losers at 6.32%.
- 📰 [S&P 500 Most Active Stocks in Today's Session: SMCI, AT&T, Ford Lead Volume](https://finnhub.io/api/news?id=f21ed546c52d7960595fde31e65328748e7d5dfb9b9a22e9d26f1190275c2277) — ChartMill：S&P 500 most-active list led by Super Micro Computer, AT&T and Ford by volume; the ten stocks ranged from a 3.65% gain to a 3.81% decline.
- 📰 [Regal Rexnord is 'Worth So Much More,' Don't Overlook Churchill Downs: Jim Cramer](https://finnhub.io/api/news?id=e957c5279b77c28aba151a9980d3dc8dc76334182bb951637b5d93e4c3d4bb7f) — Benzinga：Cramer favors Regal Rexnord and Churchill Downs, is cautious on Marvell, dismissive of Snap and unaware of Perimeter.

**Broadcom（AVGO +3.91%）**
- 📰 [What Would It Take For AMD Stock To Move Higher?](https://finnhub.io/api/news?id=290bd26bc80317b47163b24a471719adf4920a072dad69942e079bb64057fff8) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [Jim Cramer Warns About the AI Data Center Trade. What It Means for Nvidia and Broadcom](https://finnhub.io/api/news?id=8dc4b00b074d4cb970a5a270b9500e4843eccabc321cacbfd85e673f2e63ee72) — Yahoo：Nvidia sank while Broadcom climbed on the same morning, and Jim Cramer says he still won't touch either bet on AI data centers. The numbers behind his hesitatio…
- 📰 [S&P 500 Top Gainers and Losers for Today's Session](https://finnhub.io/api/news?id=6fce76dd7876e030d37b388ebacf02dc819d615b61dee883f327a61f69708576) — ChartMill：S&P 500 market movers for today's session: Charter Communications led gainers at 9.68%, while Carvana led losers at 6.32%.

**Twilio（TWLO +3.66%）**
- 📰 [S&P 500 Top Gainers and Losers for Today's Session](https://finnhub.io/api/news?id=6fce76dd7876e030d37b388ebacf02dc819d615b61dee883f327a61f69708576) — ChartMill：S&P 500 market movers for today's session: Charter Communications led gainers at 9.68%, while Carvana led losers at 6.32%.
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=7298d9b65a4f1c1d8597bf187062a57b8ac62c13c137ce3889144a6764273edf) — ChartMill：S&P 500 top movers for today's US session: Take-Two Interactive led gainers at 7.57%, while GE Vernova led decliners at 7.19%.
- 📰 [Verint Appoints Abhi Sharma as Chief Product and Technology Officer](https://finnhub.io/api/news?id=c74cf2810071f5b4e1c43a361741c0a0477300d6c5fe72ec15507339cf6436d4) — Yahoo：MELVILLE, N.Y., November 10, 2026--Verint® today announced the appointment of Abhi Sharma as chief product and technology officer (CPTO), a newly created role u…

**First Solar（FSLR -5.55%）**
- 📊 UBS 維持：Buy，目標價 330 → 295
- 📊 Piper Sandler 維持：Overweight，目標價 251 → 222
- 📰 [S&P 500 Top Gainers and Losers for Today's Session](https://finnhub.io/api/news?id=6fce76dd7876e030d37b388ebacf02dc819d615b61dee883f327a61f69708576) — ChartMill：S&P 500 market movers for today's session: Charter Communications led gainers at 9.68%, while Carvana led losers at 6.32%.
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=7298d9b65a4f1c1d8597bf187062a57b8ac62c13c137ce3889144a6764273edf) — ChartMill：S&P 500 top movers for today's US session: Take-Two Interactive led gainers at 7.57%, while GE Vernova led decliners at 7.19%.
- 📰 [S&P 500 Gap-Down Stocks: Nov. 10, 2026 Session](https://finnhub.io/api/news?id=4b6e7e9ee293a5892be5922c2159d3b2fd8682a7b7465976d0cf38d9b250a668) — ChartMill：S&P 500 gap-down stocks on Nov. 10, 2026, including GEV, APP, LEN, DHI, BBY, MRNA, PHM, NVDA, FSLR and ETN, with session moves and gap details.

**Nvidia（NVDA -5.47%）**
- 📊 Citigroup 維持：Buy，目標價 315 → 285
- 📊 Wells Fargo 維持：Overweight，目標價 315 → 300
- 📊 RBC Capital 重申：Outperform，目標價 330 → 330
- 📊 Goldman Sachs 重申：Buy，目標價 300 → 300
- 📰 [3 Magnificent Dividend Stocks Yielding 4% or More to Buy in November](https://finnhub.io/api/news?id=7dde64909cfb43722a991a075715d1a58fe995dd034910306f3de64cb4ea7934) — Yahoo：These three blue chip dividend stocks offer higher yields while standing out in terms of dividend coverage and growth.
- 📰 [Dow Jones Futures Rise As CoreWave Jumps, Lifts AI Stocks After Nvidia, Surging Yields Hit Stocks](https://finnhub.io/api/news?id=8d0b18bb07ebd921753bc040d437b96f9ea20cbe8ed9bd6481a4fa2e8c2a7cfb) — Yahoo：Futures rose while AI stocks rallied overnight on CoreWeave earnings: The stock market fell on Nvidia and surging Treasury yields.
- 📰 [Meet the 1 Stock I'd Buy With $500, Even If the Market Fell 5% Tomorrow](https://finnhub.io/api/news?id=8583ab95c01d8a72acb62051d87c1f3c5e7e777f09f4e8c01a08b6117bf8c6f5) — Yahoo：The market just had its worst day since April 2025. Here's a consumer stock worth buying with $500 anyway.

**Coherent Corp.（COHR -4.68%）**
- 📰 [S&P 500 Top Gainers and Losers for Today's Session](https://finnhub.io/api/news?id=6fce76dd7876e030d37b388ebacf02dc819d615b61dee883f327a61f69708576) — ChartMill：S&P 500 market movers for today's session: Charter Communications led gainers at 9.68%, while Carvana led losers at 6.32%.
- 📰 [AMD and Coherent: What the Biggest Gainers Since the Midterms Selloff Say About the AI Trade](https://finnhub.io/api/news?id=841222901ac783cb44e5817a841b091539fdd7c76b58e7db0d59dc6eba535370) — Yahoo：AMD and Coherent are the best-performing S&P 500 stocks since last Thursday’s AI trade selloff.
- 📰 [Microsoft Pushes Trump to Halt a Ban on Chinese Optics. Coherent Is Already “Basically Booked Out”](https://finnhub.io/api/news?id=00c6fab1d575ce4d9ff733b573ac1c86f72c239057a8b7eac66545bda4fcb1e6) — Yahoo：Key TakeawaysMicrosoft is lobbying the Trump administration to drop a planned ban on new Chinese optical transceivers, Reuters reported on Nov. 10.

**Steel Dynamics（STLD +3.76%）**
- 📰 [S&P 500 Top Gainers and Losers for Today's Session](https://finnhub.io/api/news?id=6fce76dd7876e030d37b388ebacf02dc819d615b61dee883f327a61f69708576) — ChartMill：S&P 500 market movers for today's session: Charter Communications led gainers at 9.68%, while Carvana led losers at 6.32%.
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=7298d9b65a4f1c1d8597bf187062a57b8ac62c13c137ce3889144a6764273edf) — ChartMill：S&P 500 top movers for today's US session: Take-Two Interactive led gainers at 7.57%, while GE Vernova led decliners at 7.19%.
- 📰 [Steel Dynamics Announces Additional $3.0 Billion Share Repurchase Authorization and Fourth Quarter 2026 Cash Dividend](https://finnhub.io/api/news?id=e0745721e95d7244004b3cb12468db21db474317b154e98f58465e797f45050e) — Yahoo：Steel Dynamics, Inc. (NASDAQ/GS: STLD) today announced that the company's board of directors authorized an additional share repurchase program of $3.0 billion o…

**Nucor（NUE +2.90%）**
- 📰 [S&P 500 Top Movers: Today's US Session Gainers and Losers](https://finnhub.io/api/news?id=7298d9b65a4f1c1d8597bf187062a57b8ac62c13c137ce3889144a6764273edf) — ChartMill：S&P 500 top movers for today's US session: Take-Two Interactive led gainers at 7.57%, while GE Vernova led decliners at 7.19%.
- 📰 [Ferrous Scrap Recycling - Global Strategic Business Report: Capitalize on the Market's Surge to US$163.5 Million by 2032 as Nucor, ArcelorMittal, Glencore, and Sims Metal Management Reshape Supply](https://finnhub.io/api/news?id=e3feacc422bc4a6ff9785b49c4aafe067b76db75e42efadfccc7637b50ca9358) — Yahoo：Boost ROI through EAF adoption, automated sorting, smart logistics and closed-loop recycling—cutting energy use, improving scrap purity and unlocking green-stee…
- 📰 [The Zacks Analyst Blog Highlights Nucor, UCB and Cheniere Energy](https://finnhub.io/api/news?id=9cf931e9450e62eb2cadaa6ea159ba3e31851755f391181654828d1eeb7f8e9a) — Yahoo：NUE, UCBJY and LNG headline fresh stock picks as investors weigh AI regulation, inflation risks, oil prices and global economic developments.

**CF Industries（CF +2.58%）**
- 📊 RBC Capital 維持：Sector Perform，目標價 115 → 125
- 📰 [The One Reason To Invest In CF Industries Stock](https://finnhub.io/api/news?id=57a2fd3d62eb0fa0d44e4d158b0d08410506cc4249afe2538d2ecbf7dc61f06e) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [How Rare Is Mosaic's Ten-Year Record Low Profit Margin?](https://finnhub.io/api/news?id=18d6a93e88dcf08d67bd92c92b151f06bd55fb5cbce544238438852adeb6fb95) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [CF Industries Holdings, Inc. 2026 Q3 - Results - Earnings Call Presentation](https://finnhub.io/api/news?id=47f4a32d4c3d2d9022acdb70b5e4a3697a06aab06d924481d46bf730c614dff1) — SeekingAlpha：2026-11-09. The following slide deck was published by CF Industries Holdings, Inc.

**Albemarle Corporation（ALB -4.23%）**
- 📊 UBS 維持：Buy，目標價 150 → 137
- 📊 Citigroup 維持：Buy，目標價 125 → 140
- 📊 Morgan Stanley 維持：Equal-Weight，目標價 161 → 105
- 📊 Scotiabank 維持：Sector Outperform，目標價 190 → 165
- 📰 [Albemarle Corporation (ALB) Is a Trending Stock: Facts to Know Before Betting on It](https://finnhub.io/api/news?id=b0b97fa3f8aebf821463f7470b32ef688bdd7af566c5d06c7868d0b6175e4b85) — Yahoo：Zacks.com users have recently been watching Albemarle (ALB) quite a bit. Thus, it is worth knowing the facts that could determine the stock's prospects.
- 📰 [UBS Maintains Buy on Albemarle, Lowers Price Target to $137](https://finnhub.io/api/news?id=6d276a1e94012f06820e56bc5770c2219504cd54ee15bf47defcbbbcbaaa8c3d) — Benzinga：UBS analyst Joshua Spector maintains Albemarle (NYSE:ALB) with a Buy and lowers the price target from $150 to $137.
- 📰 [Citigroup Maintains Buy on Albemarle, Raises Price Target to $140](https://finnhub.io/api/news?id=48e4db782cfc1d9477a98679e970cdf64fe7bfc0ec84f0c5a400eb3c9ef5cf0a) — Benzinga：Citigroup analyst Patrick Cunningham maintains Albemarle (NYSE:ALB) with a Buy and raises the price target from $125 to $140.

**Newmont（NEM -4.15%）**
- 📰 [Can Alamos Gold Stock Jump 50%?](https://finnhub.io/api/news?id=2e9bb27344c2d3e4316751babc98c1546fd846c5fe274423bc1631ea5739d18d) — Yahoo：For questions about this article or underlying research, email help@trefis.com.
- 📰 [Forget the ETF Outflows: Gold Miners Offer 3X Leverage As Cash Flow Surges](https://finnhub.io/api/news?id=1b4fe9ddccf23cab26cd284ee57c4ee62f8169254c65d4d848a93265a6276594) — Yahoo：The global gold market is going through a transformation. Investment in the physical bullion ETFs is shifting geographically. While the SPDR Gold Shares, the mo…
- 📰 [Summit Royalties Celebrates One-Year Anniversary of Public Listing and Highlights Accomplishments](https://finnhub.io/api/news?id=ddc3d182544eeb6e74c7a60f20d5ad14929c641127f880721d66c92a0cfdd312) — Yahoo：TORONTO, Nov. 10, 2026 (GLOBE NEWSWIRE) -- Summit Royalties Ltd. (TSXV: SUM, OTCQX: SUMMF) ("Summit" or the "Corporation") is pleased to celebrate the one-year…

**Freeport-McMoRan（FCX -3.24%）**
- 📰 [Strong Copper Prices Show Up in Miners' Quarterly Results Alongside New Capacity Plans](https://finnhub.io/api/news?id=605a01758e44a0e4a3e301c555470276babb32e74a1c28c219134e0689f3491a) — Yahoo：Energy Metal News News Commentary - Quoted London Metal Exchange (LME) monthly average settlement copper prices averaged $6.40 per pound in the third quarter of…
- 📰 [Freeport-McMoRan, State Street, Deckers Outdoor And More On CNBC’s ‘Final Trades’](https://finnhub.io/api/news?id=20a86c271ae1b0f1ae2b1db7d2ee4e47c11d32f700291d2b28b0f1d0fbd31468) — Benzinga：CNBC Final Trades: Harrington picked FCX, Firestone chose DECK, Talkington recommended RSP, and Terranova selected STT.
- 📰 [Does Freeport (FCX) Still Have Room to Run? Analysts Think So](https://finnhub.io/api/news?id=34ccd962e08abb95067ac1619707fee58dedb619796cf1915f3e7499361791ff) — Yahoo：We just published a list of the 10 Best Copper Stocks to Buy According to Wall Street Analysts and Freeport-McMoRan (NYSE:FCX) ranked 6th on this list. Copper i…

**Atmos Energy（ATO +1.17%）**
- 📰 [National Fuel Gas Has Paid a Dividend for 124 Straight Years. Can It Keep Going?](https://finnhub.io/api/news?id=adb88bb630a0e1265284f32b626bb3134f9413a985095ae255691aac5f66c026) — Yahoo：National Fuel is preparing to break into two companies, and management has not said a word about what happens to the payout afterward. Income investors face a q…
- 📰 [The Senate Voted 73-25 to Start Debate on a Bill Making Data Centers Pay Their Own Power Bills](https://finnhub.io/api/news?id=85f3a90dc034b672354a483fab6815851b6814c38084ad5cdbf444e48c88d0f1) — Yahoo：A bipartisan energy bill promises to shield households from AI's appetite for electricity, yet a wind dispute, a Title III fight, and a skeptical House could st…

**CenterPoint Energy（CNP +1.05%）**
- 📰 [Do Wall Street Analysts Like CenterPoint Energy Stock?](https://finnhub.io/api/news?id=74f112b42170bd638c4e233912b2398027c81477462a2d444209734584dd7acb) — Yahoo：CenterPoint Energy has underperformed the broader market over the past year, but analysts are cautiously optimistic about the stock’s prospects.
- 📰 [The Most Important Market Shift Of The Decade Is Already Underway](https://finnhub.io/api/news?id=2355a323b3adf31596f84792929f31a24faab7dbe44dd1535953e0ab9291d5ea) — SeekingAlpha：A market rotation could drive outperformance over 3, 5, and 10 years as investors overlook AI's long-term beneficiaries. Discover more in this market outlook.

**American Water Works（AWK +0.76%）**
- 📰 [The Senate Voted 73-25 to Start Debate on a Bill Making Data Centers Pay Their Own Power Bills](https://finnhub.io/api/news?id=85f3a90dc034b672354a483fab6815851b6814c38084ad5cdbf444e48c88d0f1) — Yahoo：A bipartisan energy bill promises to shield households from AI's appetite for electricity, yet a wind dispute, a Title III fight, and a skeptical House could st…
- 📰 [American Water Launches Enhanced Digital Experience Focused on Simplicity, Service and Customer Support](https://finnhub.io/api/news?id=5ced4e29167e7bf062742b2873a6b8cabf99d5e178413371c035bfbd57a77431) — Yahoo：American Water (NYSE: AWK), the largest regulated water and wastewater utility company in the U.S., today announced the launch of a refreshed amwater.com, makin…
- 📰 [American Water Charitable Foundation Awards $10,000 Grant to Expand STEM Learning at Putnam Museum](https://finnhub.io/api/news?id=eefabdca7018b303cd5cab095cf6e86e3d25b454855b2969107a23a8a4b9d945) — Yahoo：The American Water Charitable Foundation, a philanthropic nonprofit organization established by American Water (NYSE: AWK), and Iowa American Water announced to…

**Constellation Energy（CEG -3.02%）**
- 📰 [Surprised By Constellation Energy's Slide? What Investors Missed](https://finnhub.io/api/news?id=bca885dec94e5fbf735925e5f18070a0043d48adc0c8ab4e3a4e0a3626cc6d14) — Yahoo：If you had hesitated on Constellation Energy at the start of 2026 because bullish analysts sounded too optimistic and the bearish camp warned about nuclear head…
- 📰 [Earnings Calendar and Analysis for This Week (June 8-12)](https://finnhub.io/api/news?id=216d128b83f71d76e6bce40b8b46e11723d78dea2fbe2da4a9c9e36d4bf5b412) — Yahoo：Check out our earnings calendar for this week, as well as our previews and recaps of the more noteworthy reports.
- 📰 [Constellation Q3 Earnings Call Highlights Long-Term Nuclear Contracts](https://finnhub.io/api/news?id=2b8be770a651644cba88502fc4280a94dcac21bca9658bd5feb9fc114a32841c) — Yahoo：CEG raises its 2026 earnings outlook as long-term nuclear contracts, capacity expansion and data-center demand strengthen earnings visibility.

**Vistra Corp.（VST -2.05%）**
- 📊 Scotiabank 維持：Sector Outperform，目標價 189 → 169
- 📰 [VST Q3 Earnings Beat on Strong Power Pricing, Revenues Rise Y/Y](https://finnhub.io/api/news?id=3574d4399e1776b57d6a7c487d8e477c74f2298baac76b7d1fcdf76efe0127dc) — Yahoo：Vistra's Q3 earnings surge 84.6% on stronger power pricing and retail margins, while revenues climb 27.7%, prompting higher 2026 EBITDA guidance.
- 📰 [Constellation CEO Sounds the Alarm: Data Center Demand Was a “Massive Overestimation.” Its Nuclear Contracts Run to the 2040s](https://finnhub.io/api/news?id=60773065e80031276ab9c588b01416db54ecfb38f316ca1a17242ca7192c2995) — Yahoo：Key TakeawaysConstellation Energy CEO Joe Dominguez said utilities and grid operators made a “massive overestimation” of data center demand. Days earlier, Vistr…
- 📰 [Constellation and Vistra Get the Headlines. These 2 Power Producers Barely Get Mentioned.](https://finnhub.io/api/news?id=bdc469674424d2ac2efd8b89f783d27d04633186655e3182b142ec4a5e67977e) — Yahoo：Talen Energy and NRG Energy receive less attention than Constellation and Vistra, but both are expanding power capacity to meet AI data center demand.

**NextEra Energy（NEE -0.68%）**
- 📰 [The Most Important Market Shift Of The Decade Is Already Underway](https://finnhub.io/api/news?id=2355a323b3adf31596f84792929f31a24faab7dbe44dd1535953e0ab9291d5ea) — SeekingAlpha：A market rotation could drive outperformance over 3, 5, and 10 years as investors overlook AI's long-term beneficiaries. Discover more in this market outlook.
- 📰 [NextEra Energy Is Trading On The 10-Year While Its Backlog Climbs To 36.7 GW](https://finnhub.io/api/news?id=26c5155a9b7ed5695a35f7a49838ee402cad35b1168221055b1053c732f29058) — SeekingAlpha：NextEra Energy delivered 9.7% YoY adjusted EPS growth in Q3 and reaffirmed guidance, with NEER's backlog reaching 36.7 GW. Read why NEE stock is a buy.
- 📰 [UTF: 7.7% Monthly Income At A Discount Again](https://finnhub.io/api/news?id=68f597bf787ab80924491fb31388fe49fd41a96b55843d1758925a7c61efc0bd) — SeekingAlpha：See why Cohen & Steers Infrastructure Fund earns a buy rating: 7.7% yield, NAV discount, and covered payouts through 2026.


**當日財報｜2026-11-10（S&P 500 成分股，依營收規模排序）**

| 公司 | 時段 | EPS 實際 / 預估 | 營收 實際 / 預估（億美元） | 當日股價 |
|:---|:---|---:|---:|---:|
| TransDigm Group（TDG） | 盤前 | 12.44 / 12.33（優於預期） | 29.9 / 29.8 | +0.55% |

盤後公布的財報，當日股價尚未反映，實際數字也可能稍晚才更新。

## 6. Treasury Yield Curve

In [6]:
def treasury_curve():
    frames = []
    for yr in (AS_OF.year, AS_OF.year - 1):          # 年初資料不足兩天時補前一年
        url = ("https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rates.csv/"
               f"{yr}/all?type=daily_treasury_yield_curve&field_tdr_date_value={yr}&page&_format=csv")
        r = HTTP.get(url, timeout=TIMEOUT)
        r.raise_for_status()
        frames.append(pd.read_csv(io.StringIO(r.text)))
        if sum(len(f) for f in frames) >= 2:
            break
    df = pd.concat(frames, ignore_index=True)
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
    return df.sort_values("Date", ascending=False).reset_index(drop=True)


def curve_shape(d2, d10):
    """依 2 年、10 年殖利率變動（bp）判斷曲線型態"""
    if abs(d10 - d2) < 1:
        return "大致平行移動（2 年與 10 年變動相近）"
    if d2 <= 0 and d10 <= 0:
        return "牛平（長端降更多）" if d10 < d2 else "牛陡（短端降更多）"
    if d2 >= 0 and d10 >= 0:
        return "熊陡（長端升更多）" if d10 > d2 else "熊平（短端升更多）"
    return "扭轉變平（短升長降）" if d2 > 0 else "扭轉變陡（短降長升）"


def fred_curve():
    """備援：FRED 的公債殖利率（資料同樣來自財政部），5 個天期同時抓"""
    if not FRED_API_KEY:
        raise ValueError("未設定 FRED_API_KEY")
    cols = {"3 Mo": "DGS3MO", "2 Yr": "DGS2", "5 Yr": "DGS5", "10 Yr": "DGS10", "30 Yr": "DGS30"}

    def series(sid):
        r = HTTP.get("https://api.stlouisfed.org/fred/series/observations", timeout=TIMEOUT,
                     params={"series_id": sid, "api_key": FRED_API_KEY, "file_type": "json",
                             "observation_start": f"{AS_OF - timedelta(days=30):%Y-%m-%d}"})
        r.raise_for_status()
        return pd.to_numeric(pd.Series({o["date"]: o["value"] for o in r.json()["observations"]}), errors="coerce")

    with ThreadPoolExecutor(len(cols)) as ex:
        df = pd.DataFrame(dict(zip(cols, ex.map(series, cols.values())))).dropna()
    df.index = pd.to_datetime(df.index)
    return df.rename_axis("Date").reset_index().sort_values("Date", ascending=False).reset_index(drop=True)


try:
    tc, source = treasury_curve(), "美國財政部"
except Exception as e:
    print(f"財政部資料無法取得（{safe_err(e)}），改用 FRED")
    tc, source = fred_curve(), "FRED"
cur, prev = tc.iloc[0], tc.iloc[1]
rows = ["| 天期 | 殖利率 | 變動 |", "|:---|---:|---:|"]
for t in ["3 Mo", "2 Yr", "5 Yr", "10 Yr", "30 Yr"]:
    rows.append(f"| {t} | {cur[t]:.2f}% | {(cur[t] - prev[t]) * 100:+.0f} bps |")
d2, d10 = (cur["2 Yr"] - prev["2 Yr"]) * 100, (cur["10 Yr"] - prev["10 Yr"]) * 100
spread, spread_prev = (cur["10 Yr"] - cur["2 Yr"]) * 100, (prev["10 Yr"] - prev["2 Yr"]) * 100
display(Markdown(f"**美債殖利率｜{cur['Date']:%Y-%m-%d}（vs {prev['Date']:%m/%d}，{source}）**\n\n" + "\n".join(rows) +
                 f"\n\n- 2s10s 利差：{spread:+.0f} bps（前一日 {spread_prev:+.0f} bps）\n- 曲線型態：{curve_shape(d2, d10)}"))

**美債殖利率｜2026-11-10（vs 11/09，美國財政部）**

| 天期 | 殖利率 | 變動 |
|:---|---:|---:|
| 3 Mo | 4.30% | +6 bps |
| 2 Yr | 4.73% | +17 bps |
| 5 Yr | 4.81% | +14 bps |
| 10 Yr | 4.98% | +10 bps |
| 30 Yr | 5.35% | +9 bps |

- 2s10s 利差：+25 bps（前一日 +32 bps）
- 曲線型態：熊平（短端升更多）

## 7. Free news

In [7]:
CNBC = "https://www.cnbc.com/id/{}/device/rss/rss.html"
PAYWALLED = ("reuters.com", "bloomberg.com", "wsj.com", "ft.com", "barrons.com", "marketwatch.com",
             "economist.com", "nytimes.com", "investors.com", "washingtonpost.com")


def google_news(query):
    return ("https://news.google.com/rss/search?q=" + quote_plus(f"{query} when:1d")
            + "&hl=en-US&gl=US&ceid=US:en")


GROUPS = {   # (來源名稱, 網址)
    "盤勢與個股": [("CNBC", CNBC.format(100003114)), ("CNBC Earnings", CNBC.format(15839135)),
                   ("CNBC Investing", CNBC.format(15839069)), ("Google News", google_news('"stock market today"')),
                   ("Finnhub", "finnhub:general")],
    "總經與聯準會": [("CNBC Economy", CNBC.format(20910258)),
                     ("Fed 新聞稿", "https://www.federalreserve.gov/feeds/press_all.xml"),
                     ("Fed 演講", "https://www.federalreserve.gov/feeds/speeches.xml")],
    "美債與美元": [("Google News", google_news('"Treasury yields"')), ("Google News", google_news("dollar forex"))],
    "黃金": [("Google News", google_news('"gold prices"'))],
    "原油": [("CNBC Energy", CNBC.format(19836768)), ("OilPrice", "https://oilprice.com/rss/main"),
             ("Google News", google_news('"oil prices"'))],
    "中文（鉅亨網美股）": [("鉅亨網", "cnyes:us_stock")],
}


def fetch_items(name, url):
    if url == "finnhub:general":
        if not FINNHUB_API_KEY:
            return []
        return [(datetime.fromtimestamp(d["datetime"], timezone.utc), clean(d.get("headline"), 300), d.get("url", ""),
                 f"Finnhub / {d.get('source', '')}", clean(d.get("summary")))
                for d in finnhub("news", category="general")
                if d.get("datetime") and not any(p in d.get("url", "") for p in PAYWALLED)]
    if url.startswith("cnyes:"):
        r = HTTP.get(f"https://api.cnyes.com/media/api/v1/newslist/category/{url[6:]}",
                     params={"limit": 50}, timeout=TIMEOUT)
        r.raise_for_status()
        return [(datetime.fromtimestamp(d["publishAt"], timezone.utc), clean(d.get("title")),
                 f"https://news.cnyes.com/news/id/{d['newsId']}", name, clean(d.get("summary")))
                for d in r.json()["items"]["data"]]
    r = HTTP.get(url, timeout=TIMEOUT)
    r.raise_for_status()
    items = []
    for e in feedparser.parse(r.content).entries:
        t = e.get("published_parsed") or e.get("updated_parsed")
        src = e.get("source") or {}
        if src and any(p in src.get("href", "") for p in PAYWALLED):
            continue                                   # 略過付費牆媒體
        title, publisher = clean(e.get("title"), 300), src.get("title", "")
        if publisher and title.endswith(" - " + publisher):
            title = title[: -(len(publisher) + 3)]
        items.append((datetime(*t[:6], tzinfo=timezone.utc) if t else None, title, e.get("link", ""),
                      f"{name} / {publisher}" if publisher else name, "" if publisher else clean(e.get("summary"))))
    return items


def merge(fetched):
    """同一組各來源依序合併：只留最近 NEWS_HOURS 小時、標題去重、新到舊取前 NEWS_PER_GROUP 則"""
    cutoff = datetime.now(timezone.utc) - timedelta(hours=NEWS_HOURS)
    seen, items, errs = set(), [], []
    for name, (its, err) in fetched:
        if err:
            errs.append(f"{name}：{err}")
        for it in its or []:
            key = re.sub(r"\W+", "", it[1].lower())
            if it[0] and it[0] >= cutoff and it[1] and key not in seen:
                seen.add(key)
                items.append(it)
    items.sort(key=lambda x: x[0], reverse=True)
    return items[:NEWS_PER_GROUP], errs


# 所有來源一次同時抓（原本是每組內逐一抓），再依組別合併
tasks = {(g, i): ("news", partial(fetch_items, name, url))
         for g, sources in GROUPS.items() for i, (name, url) in enumerate(sources)}
res = gather(tasks, {"news": len(tasks)})
results = {g: merge([(name, res[g, i]) for i, (name, _) in enumerate(sources)]) for g, sources in GROUPS.items()}

lines = []
for g, (items, errs) in results.items():
    lines += [f"### {g}"]
    for when, title, link, src, summary in items:
        title = title.replace("[", "(").replace("]", ")")
        lines.append(f"- `{when.astimezone(TW):%m/%d %H:%M}` [{title}]({link}) — *{src}*" + (f"：{summary}" if summary else ""))
    if not items:
        lines.append("- （無）")
    if errs:
        lines.append(f"- ⚠️ 無法取得：{'、'.join(errs)}")
    lines.append("")
display(Markdown("\n".join(lines)))

### 盤勢與個股
- `11/11 11:45` [Stock Market Today: Sensex Jumps Over 1,100 Points, Nifty 50 Opens Above 25,300 On Lower Oil Prices, FII Inflows](https://news.google.com/rss/articles/CBMi5wFBVV95cUxQeTYtODdweFhZS2g5dDJRMGlWWl91dFozRmRBdTFSeDE0Ty1waXgtRUNVQUllVEl0dGpFMjYzZE5QeFU3Mk9KZ0c1OUI2cXYwUW1OcS0xWkNQR1J6WC16SUhsdXJZWjJKNGU5cXVhSmoxRllBQzZkZU5PbDRLMEtDUnhJV2hmU185X2pmY2ExWWpXeWliSzhJRFJWb1BDclN1dS1zXzQwNHpJamZWUnZaOWdiM0NBMi1WRW16YjNtQV8tQW9VU1Vtc3dXXzZRdFBNcGltV2o5UjV4eUN3ZEY0WERyZXhoWG_SAe8BQVVfeXFMTS1nc2N5SjYyWml1NzBGVHJzcUc1dFZlMmRHZFFxTGhNSlpzX0NMN3V6d2U3c09HSHJ2eUlfdXdnSVJfTE9uZ1dPWmMza0p3TEE3Uld1MGZkWUQyVG15M0ZvNGppS0hBQVljcTRURDJSZUV4V0hER21aRXZMMDlwaU9yaGhpcWc3NHJsdE9STm1Ja2FsTHV6aFRYY3dpc2tzQTFRR2g4dHdWSEtsNE1JZnhmdlY3dGpWbUtzbkUxZWw4V3hYS1ZTUUs1LUtCU3htbWZleVg1VTBJM1d2eURQUmRiZnQ2TktzRGFYWTQ2NE0?oc=5) — *Google News / NDTV Profit*
- `11/11 11:08` [Alibaba announces plans to list T-Head, its Nvidia chip rival](https://www.cnbc.com/2026/11/11/alibaba-pr.html) — *CNBC*：Alibaba noted it had already filed confidentially on Sept. 11 to list T-Head on the Hong Kong Stock Exchange.
- `11/11 10:46` [Singapore wants a bigger slice of the global gold market. Is it too late to the party?](https://www.cnbc.com/2026/11/11/singapore-gold-market-vault-storage.html) — *CNBC*：Singapore is stepping up efforts to become a key player in the global gold market, betting that vaulting service and tax incentives will help it attract flows.
- `11/11 10:10` [Oil rises from two-month low as Hormuz shipping attacks offset Iran truce hopes - Reuters](https://news.google.com/rss/articles/CBMiwAFBVV95cUxNUDdYdlFvTWpsUDJOZkVpN1RNYmM1RlIwZUxzSGhrazVnMTJ1el9lT2lpc08yc2VrUjdyR0tSWjJIRVRMdXh2cmJDam5yY2d1QURuUFdvQkh1ZWJDdHFXbmZCejViSHhTYnl6YU1uMmEtS1NIcFU1dmhPQVZpN2R5Ykc1OVBrUXVhemRVa01TblQ5THN6eDBlOXl3Nm5qTEtKRjNDeVVraW1hRjUwZjQyclBwd1hCTlZvNkdaVVdlUDc?oc=5) — *Finnhub / Reuters*：Oil rises from two-month low as Hormuz shipping attacks offset Iran truce hopes Reuters
- `11/11 09:59` [COST Stock Rises Overnight: Costco’s $30 Special Dividend Has Retail Celebrating ‘Santa Coming Early’](https://news.google.com/rss/articles/CBMinwFBVV95cUxPYU03ekRHaEc0TWl0STFiVExmZnBzd0xRTXdpa2ctc05XeS1qRWlvb3J1NFdaMHVFc2lsLVNkeG1meGZSZTN6MUlLeTRNZFI3c19JeEhWbTlFdFV3MkxJWkt3a05iWVp3ZnZYanZXSTNWV21pLVZ6NTFrT3ZFYmZSOEVFN2pONzdlNHdSOHFYQTJJVzkxeWZQYmZjSlpJMmc?oc=5) — *Google News / Yahoo Finance*
- `11/11 09:52` [Anthropic opens Australian retail IPO offering as AI scrutiny intensifies](https://www.cnbc.com/2026/11/11/anthopic-australian-retail-ipo-offering-.html) — *CNBC*：Anthropic opened its Australia retail IPO offering on Wednesday, allowing individual investors to apply for shares ahead of its planned IPO.
- `11/11 09:29` [Stock Market Today: Gift Nifty, US Inflation To Oil Prices; Five Key Factors For Sensex, Nifty 50 On Nov 11](https://news.google.com/rss/articles/CBMi9gFBVV95cUxOWkpfWmtRTTJKWG1RejBvZ1owRldmMG4tSW43bGEtOHAxRDdzOXNuRXo2R0FxVWEtb0NoZktRanUzZzMteFh3YUlNb040MlZKbEp2LTc3aTVQblVnVTlMNUFGMWN4ZUtGUVpJelhoQTh4OWszdldubVB5NTQ5VnpKVDZQanF0bGRrMEl1Tk0waVNXU285eEhjaS1rdFNWVllzUGdIVG1nNEtIZFlHZS02S0JwN2taSzU4TVYxVHM1Y3o2dFhRYlN6RVp6dmM4M085RkJXVVRtVnpLZDVGdGVIM2JScC1oVnYyZ1pfZEZlOHlYMW5NOVHSAf4BQVVfeXFMUFEzWWFxUnBpNG9PV2pzTy0zRllqd2ozZzFQeU5CY096a3J0eThvOGtkLWlXelZBY1NfWVpDRmhfTS1WdGdSWXM0RnpXZlRLNy1feXFLZkpCWjNLNHZsVEVVZ1pkVGZGRGp3YkkyLVBCeXI2a3NQbXRoRm95UDNiR3JOQVNkTURxeWRuXzVtT2RIRWxmRkp0ckIwUzc2XzZEQkdaSVp0dmRYdkl4X3ljV2psYWE5T0kxQnF0MnFMZ1p6ZEMteU12ZjhSSWI0SmdPYjdKU0QxOGVQaXF1V3Bab3RrODFfek04N21Ham1IcWxSUjI2Z2dOQUxJa0tZMlE?oc=5) — *Google News / NDTV Profit*
- `11/11 09:15` [Stock Market Today: Dow Jones, Nasdaq, S&P 500 Fall on November 10, 2026; Oil and Gold Retreat](https://news.google.com/rss/articles/CBMizAFBVV95cUxNTnAwU2t5T2szQUhFeXdXUVk0WnZ2Qi1hanhRcnZsaVk3VEpKSmhFeFI5Y3FiOXdvN2dtbEZlZkdCZW9aWDlUOGZaS2t2aHJ0bHZCMFdSTmxXWjhRMExMWmJCODhwQVlvV3gtckVNZm9nREVLVE9GaDFNUXJMVzVLTXJPWk85aUhheTlRT3ZGanNNSW44RFNXTVR5R1FuUTJPQUItSXZBT1UzSEtRVmxMUldGeXJyZ0FGczFsNDR6QXRzMDZJUW1ibklwRWg?oc=5) — *Google News / Eurasia Business News*

### 總經與聯準會
- `11/11 07:53` [Consumer prices rose 0.4% in October, more than expected, complicating Fed's inflation fight](https://www.cnbc.com/2026/11/10/cpi-inflation-report-october-2026.html) — *CNBC Economy*：The report comes at a critical juncture for the Federal Reserve as it considers its next move regarding interest rates.
- `11/11 04:30` [Bowman, The U.S. Economy and Considerations for Monetary Policy](https://www.federalreserve.gov/newsevents/speech/bowman20261110a.htm) — *Fed 演講*：Speech At the C. Peter McColough Series on International Economics Council on Foreign Relations, Washington, D.C.
- `11/10 18:24` [China's goods surplus tops $1 trillion for a second year amid calls for Beijing to rebalance trade](https://www.cnbc.com/2026/11/10/china-trade-october-exports-imports-surplus-.html) — *CNBC Economy*：China's trade surplus topped $1 trillion as of October, potentially intensifying complaints from trading partners already wary over the glut of cheap Chinese goods.
- `11/10 17:00` [Waller, The Economic Outlook and Some Thoughts on Labor Market Dynamism](https://www.federalreserve.gov/newsevents/speech/waller20261110a.htm) — *Fed 演講*：Speech At the Bank of England Watchers Conference King’s Business School and Money, Macro and Finance Society, London, United Kingdom

### 美債與美元
- `11/11 11:27` [Currency Exchange Rates in Pakistan Today – Latest US Dollar, Pound, Euro, Dirham Forex Update - 11 November 2026](https://news.google.com/rss/articles/CBMi5AFBVV95cUxNeDVXcFFtaFE4Zm04SUw3cHhraEd6cEduQmVLRWdfLXU2UXkzVXNsZ0I2MXFYR1gwT29TUmxTbTdoelZBaDVNcWVneWd1dHB5MDdOQVhIMkp6Unk5Sk0tZHp2bjJDTVNWSFdlNi15WlZTYVVKX1ZxV1RwVGJEalBGanZSWEVzTW9LM3JQZFljQ3Zpc1BmdnBTQ1puanRmb0plNU9XV3FkVnBYYzNodnJjOF96MWw2dW9JcEVBTTJhQ1pvYUg1Q25DTE82YXNQTW9BQ1JHZnZHNm5UZEFwWjNOa0cyaC0?oc=5) — *Google News / Daily Pakistan*
- `11/11 11:09` [Treasury Yields Jump as Hot Inflation Fuels Fed Rate Bets](https://news.google.com/rss/articles/CBMikAFBVV95cUxPUVYxRy1TeEsyWmxwSWtoVTZiS181b0R6OXZYNVBkQ0RGTHE2cjFlRmZaQkxfVVNFNzJMdm9nWVdNUUwtX1JjZmZUWUtSZ2lpYUt6MGFRWXBoOF9BOVZDUTlKUnk5QkR4Z0VZajY3eTRvS3Q5Z3pibWVVWFdEX1BFT3pyMF9SUWJnd0VCblFPbmQ?oc=5) — *Google News / JournalArta*
- `11/11 11:07` [U.S. core inflation stays sticky in October, boosting rate hike bets](https://news.google.com/rss/articles/CBMiYkFVX3lxTE1fa1UwWHdNc3JvRHlYdlRacHZGcjlCenpReWEwREJMSS15cHpfMTFQS1JKaW5zZG9HTllPTjBZaFM3MXllX08yZnR5amlFUmlhaHFtMjlhaHA1dEk4ZS1wdE5n?oc=5) — *Google News / Global Times*
- `11/11 11:00` [Korea FTC fines 15 financial firms 1.6266 trillion won for bond collusion](https://news.google.com/rss/articles/CBMif0FVX3lxTFBzTWs0TFRXRUtYem9LcHZ5aG5TYWpObTFGeXFrN1RnWGhQa3FweE9QVEl0SWZkN3BPWUFtUjhVWXJSd3FQeFRwMUp2U2xfdUFrNm1ZUmtoUkt0ZFFBVTU2anFEcjY2QkZ5cTFtczJwd1JnRHdWN1NsVVlGbVBRNTTSAZMBQVVfeXFMUGMwVEl6OTdhZ2RMbzNSXzhTRUhUbC1hbFBPNG1GX2syXzlsV0w5TXFpTDI2VkEteDByVlVxUkFVRFRNVEF6Q0ZfbFVwa01sbElJVEhNcFIyMmdXVGJ5NXBfMkJ4dGdVcWFhazE0a292RjgtZ3hPaUItdVQxbU9LX0dWc1dBcWZhVjh0djFWcTVuMFNZ?oc=5) — *Google News / Chosunbiz*
- `11/11 10:57` [Why is the Canadian Dollar struggling despite higher oil prices?](https://news.google.com/rss/articles/CBMimAFBVV95cUxPZmlVUEktUXBpb1ZOUjVMckZMb0lVeWstZnVadW53NTFjYUd6cnRmZnNwcV9mYjBJQS1BMlE4UkVTTTRRTlo3V2dPX1lQQXM0dnZjZ0J1RTVkOGhCNnJYc0JfVTZpcXA0cjVjczI3RVBHSjZuNE84MHo0Tm1lTVpORDNoc0FGcDBSNDZIakxRbVRVNXFXS0hnMA?oc=5) — *Google News / FXStreet*
- `11/11 10:14` [Pre-Market Update: GIFT Nifty Signals Higher Opening; US Inflation, Treasury Yields and Q2 Earnings in Focus](https://news.google.com/rss/articles/CBMi6gFBVV95cUxNT0YzM3BfQmpJcWpGX1Zvclg1SW8yb2dJYlBmNnVfVWpUb25hSmhCTDNZUTVvQlhtTnFqMWRtUm1LdUh4WnRHYUdJQ0pnYzNZOEJmMm40OTV2M2hCVXQ4cG8wVU13X2JIQmx4NUdBeHJ6dlJHUkxQN19obEF3MVpzSjBUNnUxLW1oZFk4YWhfaHhzOW0wTzFPWjVrYlVUeDJOd3p5dUFyTnZrVWJTaWs0V3lrRzZBdzFSMDhBb3RQWkJHM3UxNFRaNU1zYVJMYjNBMWxfOFZwOE1QZGR2ejVfMmZxV3dseTAwdWc?oc=5) — *Google News / Dalal Street Investment Journal*
- `11/11 10:02` [Uranium Energy's Spot-Linked Supply Deals Face a Rates Test](https://news.google.com/rss/articles/CBMiywFBVV95cUxNQ25rcm41MzAxNHNoTEtsUzFWTGllX1ZiNG5WcHp0QXM2WDU5RDBDQUZFWFFlM1J3U2xKS1RTYnVWYXJObDhhWDZfNS10Z0lac3NOdFlUcmh3V3R4WE9kNTV2MjdiNXQ3ZFpiMUpnVmUwQW9NaENzd3FuaUJxcUtfeUVIU2M0LTc4WElEenZNanc3V0U5eklKX1RFRVlsM01hUjd1czZ4UFV1aVg4bVRRNTYwdFZ4MTBkYmN1dndTMVZMc0ltckVCeE9EWQ?oc=5) — *Google News / ad-hoc-news.de*
- `11/11 10:01` [Japanese Yen struggles as Fed rate hike odds lift US Dollar](https://news.google.com/rss/articles/CBMiogFBVV95cUxOWFhzY1M0ZUNSS0dLNnY5cm5pUmNJLWRPbWlFQnV6VkVIRndUOUU5eGR1VlVpaUJxcnZzbHBLLUZubVF1TlR3NUJORGtjaUFBcUJqTk1UVXpoXzczclpCNjFmMVo1VGxCM0VvYzdabkJqT0lLakh6cmk0cl9yLTNJazFBOXUybEozYnRBNHBmcWNNbzZ2NFVFLTZ3U0VhOG01MUE?oc=5) — *Google News / FXStreet*

### 黃金
- `11/11 11:24` [Gold prices continuously decrease, buyers lose up to 13.8 million VND/tael after 4 months](https://news.google.com/rss/articles/CBMivwFBVV95cUxQeW9KM204eDdsQmZxTFVJd29PQzJyTWFwa3hvQUZiVldtOFo3eVZyZHM2dDV3WGhpaUJjMFZIalRjZVY5azhCY3p6Y2kyMUpMU1pGTkMxdWFwdUJVT0dkdFVMUDNULXkzd2thUG9yWnhiNFNxZ0xxMG05d2dHRWtQTExZYU52RTRwLU8yZk9QTDFhMFN2MlZQaVVoblJWZ0tIck5GZXpHcVNkTF9BVERHdXpwcVdndm9lMVBIakNOUQ?oc=5) — *Google News / Laodong.vn*
- `11/11 11:22` [Oil rises from 2-month low as Hormuz shipping attacks offset truce hopes](https://news.google.com/rss/articles/CBMi3wFBVV95cUxOMEdSSmNjd0lqbU5KZ2pmekdvMkhlTFcxQTg2N0Z1YjlFZDd1V2E1am9DUDZtUHZaaDVKZXRqLWhLd2FnU2NzaGZsVnl4blVqeUY1OXA4bVdNdkt0Um1lMUpDUE5jQ3NFb1VlTWVPT0xCTkRhM180R0ZlSDBfYkxLSzFveElXYnd5MGNJeGpNMjd0VkZfamR2RnlTaTFIT2JoVTgzS1djRXN5b3lhNXVPd1Fab1FKT2dUcGJVcDFBTEI0UmdyQmVQYTlMQ3RuRTBRdUtDWXExcVM3dnZDdFVZ0gHkAUFVX3lxTE1LYWhxdmJWd3UwbE1xakdyN0NSNXpOUURCUGZFM0kxQm9EUlk3d1hoNVVJelhlakl3bFNmVF9hWGtNcXRibTJBYmczbUJ2R0hhbXZOMWFGaGhtamo4VkRyd2N0SzY5Wk1iU2pCWmxnN0pPTzV4SVhXcC0xcm42YUQ4N0xKVXh0RS1Ed0xzX29QbWdQUDhyZWRaZmRYVDJXZzJFcXRxcDZ0V21YdUxGRlBQeThuMzN1LUlQQnNFdkVsQ19ka0U2d2ZVVEhQWGhuSFk5V1ZCTlBkbnU5Mng1MUVCRGlzeA?oc=5) — *Google News / Business Standard*
- `11/11 10:59` [SJC gold prices fell by 1.4 million VND/ounce on the morning of November 11th following US inflation data.](https://news.google.com/rss/articles/CBMiowFBVV95cUxPUzJpbDVzQ1pTallIdDh2SVdiYVR0anhfSl81NFQyUFpaR3ZINzRFTHRNNU1sTFFiclZXazYwMzh3dmdKWjVWa24zU3dfLXk5WGhVZkZaRVI0RDRIeklaSlNMX1dfWVlMckFDdjBORTB0c3lHbkNQODFKQlVIOG1hQlRkNmRXdWM2VFVpUWVVcDFjSEVybEhQTjJVcDg4ZVkwOTBv?oc=5) — *Google News / Vietnam.vn*
- `11/11 10:48` [Newmont Stock Up 26% YTD: Should You Buy, Sell or Hold the Stock?](https://news.google.com/rss/articles/CBMi4AFBVV95cUxPelVpdzVQa3pETjF2TXplTlpNWDZxdzF6M25QT1RRS1Z4eG9iRTJ1bEJoWGlyRHJHZDRFSkZuaS1JWVNLRExFZzNYZG5nNTM5R2FlcFp5bmlvVndYb1BPY0lxRjNaWVVmeUFVejk5TXl6UlpuNUVsa2NqVkJzX1BaWjBPVTZOaHIxZDZ6em9rcjV2dTBtVWlZVHh3Y3hDNHpHQmthOTNOU2ZieFZxTm9rM0c1RXFteWdjX1l3UDZSaEtRUjBralpPOXNWTktweU9MUFl1VWxYNkhIRjZneFVyQw?oc=5) — *Google News / The Globe and Mail*
- `11/11 10:43` [WGC assesses gold price outlook after deep drop](https://news.google.com/rss/articles/CBMiogFBVV95cUxNMnlLVmFZZjctV3FEclIxeEdGTV9qdzdJVkg5VTlxSkhObWxhQ1FWMEttU2xmOXduOXR6aVRqV0t4ekhaZ3pEOWhiQnNCWU5PNkh5VU03OHZmRWZqTVZOU2RKcmJsd2VhLVlpZm05aWp1bVRMdEY2TjZQa3NnbFJZOEZjWE55dEhOSUYzY29LNGttUzdiMDBLdl9JZW9aaUpOVUE?oc=5) — *Google News / Laodong.vn*
- `11/11 10:37` [Gold Tumbles Toward $4,000 as Hotter-Than-Expected CPI Boosts Fed Rate-Hike Bets, Bulls Eye $3,975 Support](https://news.google.com/rss/articles/CBMi6AFBVV95cUxNdVZqY1YwX21KSVpVNWhFOG9JekRPa010WGRYd2RTNUFRRlBEYW0zVjFwYjFjTm1wRG1jbDNuYzFZZ3F4X3V1cXJHZ2VfaG1qaWYxRGtEV0lSdEFycDNZdnRhME13TWY1OU9rc0lsWGxxZHR4OWd4NlcxcHB5b2d3anFCeTVqcGRFYUJXajhZVFdEQnIwQThnU21CSU5KMDJaTUZCYk5DWklyTE1DM2JJelZRbWtfQTZ5bEFUZnVDX19RWjhfQzMyYzJFY3JvYlFzdnRqZDNUY3lGdS0xSGtZcUh1cVJ0TnBl?oc=5) — *Google News / TradingKey*
- `11/11 10:28` [Gold and silver price today, November 11: Check rates of 24K, 22K gold in Delhi, Mumbai, Kolkata other cities](https://news.google.com/rss/articles/CBMiigJBVV95cUxQb3RrQTBFcnhUYTQyTVRka2s3N0JLMGpmX2tlVE1VVzZDZFUzTW1xLXR0ZG4zU1BlNHBxLV85UWdPT2gyaXphTk5PNEdzZXdweVR2WGQ5NGV3Q25JV2c5YlZpUUdkNnVwYWp0ZkJwMDI3TTFvOUV5dlVyY2ZiSklYZlZhSXA3YmhxWWQzbEFzWWlQOTJRTGQzMXJsUk9SdEl5eFk3UmYtVkl3emFEUU5LSjFpbHhUUFAzVGlQd2xuOHhJSmN0ODN4azZLZWx2TzNraFRGV29aS1NfcUZ5bVhzWFFfbjhsZUVNQS1sRUJ2ckNSYXlYSWlPRUdzTWZXS19Pd3RzY0o1VU12Zw?oc=5) — *Google News / Moneycontrol.com*
- `11/11 10:14` [Gold, which is considered an asset against rising prices, has fallen since the announcement of the U..](https://news.google.com/rss/articles/CBMiT0FVX3lxTE1aTFdHYkJMWGF6a1dmLTVCa0taMlRfVkNYSzFVRzZUX0ZoYk1MZndSYnpZV0NaQkpOM24zVmFtRlFBR3Z3UUFaTnJFTVE0dXc?oc=5) — *Google News / 매일경제*

### 原油
- `11/11 11:00` [Why mainland China investors are finding better returns abroad than in Hong Kong](https://news.google.com/rss/articles/CBMi2wFBVV95cUxQMnFJSDlJaGRXUkdkcF91cU1uMVVXaUdWMnMwcGZlYTJsendveWNTUFV5b29sekU0NEgxcjJOZW9YSEkySUNBeXVDTV9zOUdmTm5kTkNoblh1UG5GS3M2dVRPcEF0QkNScVhxeXI4TDlSN1ZtTEh6cmw2QnVQdVdrN2VsSjYxcDZGMnVwNW1jOVR0LTliVlp6TVRfRlRGX0xHaGNUZ2lWYmthYmFqeTJxWkxxOHFvTHE2c00wTlVLdDhueTdiUFZJNGxaQUV1M2h6RFJCMm8xN0RPVWs?oc=5) — *Google News / South China Morning Post*
- `11/11 11:00` [India’s Rupee Finds Room To Rally As Oil Prices Ease](https://news.google.com/rss/articles/CBMihwFBVV95cUxQcktzUnhHNWM1SGM5YThGSU94TFZrazZSNDlMNWxieVpsQWlISFR0OVFOMkgwaVNWREplbVJjVG9nSlJHeUpiLTR4Wm5XVkJLOXZEdTM5VV84WXJKMFdPV09mMW1NX0lDdG9rMW0tSGJFNnFKemdzWGJOak0yV2tpNC1aUWJjUEU?oc=5) — *Google News / Finimize*
- `11/11 10:57` [Why is the Canadian Dollar struggling despite higher oil prices?](https://news.google.com/rss/articles/CBMimAFBVV95cUxPZmlVUEktUXBpb1ZOUjVMckZMb0lVeWstZnVadW53NTFjYUd6cnRmZnNwcV9mYjBJQS1BMlE4UkVTTTRRTlo3V2dPX1lQQXM0dnZjZ0J1RTVkOGhCNnJYc0JfVTZpcXA0cjVjczI3RVBHSjZuNE84MHo0Tm1lTVpORDNoc0FGcDBSNDZIakxRbVRVNXFXS0hnMA?oc=5) — *Google News / FXStreet*
- `11/11 10:43` [A Venezuelan Billionaire Just Shared The Details Behind Trump's "Biggest Oil Deal In World History." Here's What It Means For A 68 Year Old With A $2.4M Portfolio Heavily Invested In Energy Stocks](https://news.google.com/rss/articles/CBMi1AJBVV95cUxQekV0dC1PLW1HV3lMQk04c0tSaHRJU0s4VXFfVXB1a2VPMkE5MVdzMklEeldQa1hvV2ppdmZtR29DbEZBWVc0QWFGbVItRmo4QW9WWkprbE9PY3dtb2JyQWoyWWJjSU85M1h3UmpsTms4VDVmTmFyWnNnZWlkMjVTMUF2YlVRODNJRlZTLVpfVHlzVHlJVzY3blA5aXdKQU5nRWkxTVI0TUVCYU9qWmZzTS1nV3ZEQjZHY2NfYXp6ME5YTV8xT1R3bVRuMkl0elFwSEk0eWh2YjR4RkFpT0hEd0lqOHIxMjBOblUxdGROZVJyb2tRRDdfTG5LTFAwRmlNYVAzaHZMYWJEYUpFNEVmWXBwdzBWOWpobjU1bDlVZ3IzcXdZWTdKMXo4dEJJajZEVjNFTVp0ZEFGS2xvbEdIRExYcGwzdkJ1SVc1NTN2MWxfZEZu?oc=5) — *Google News / 24/7 Wall St.*
- `11/11 10:42` [Oil Prices Recover From Two-Month Low Amid Fresh Hormuz Shipping Attacks](https://news.google.com/rss/articles/CBMiuAFBVV95cUxQTm9RRTBLUFl5VlVzN0xaMUhXcWtuZlpPTkFUQ0RudDkzay10VERkZWpTSnRoVjRmUEtvOUozUkttdFZCZEJZQVR1WXh3ejhBYnpzQkFXeXMzUDc1UkhDbWN1dTRsMTE5Ym9kWVFCOXEwNXI0MnBhWmNMdW1nNE5lYzZOVnllTTZuWUlSbEdXR1NGbWFQdGo4NUlXUUFpeWNBLW1PRUlFMzR4MGwtWlcxNDU5T0FJM1Et?oc=5) — *Google News / BusinessToday Malaysia*
- `11/11 10:40` [Home heating oil prices about $2.50 higher than last year, EIA says](https://news.google.com/rss/articles/CBMingFBVV95cUxQTWpaaDBZMzBmTV9SVGY3Rnc3ckdpbFR5MU9qd3pybDZaYktZWnFfWjVJem56Yl9mSFVpMFNheEtDZWp2TUFobHl2NWFDZWE4Mm5VTFQyaEhrbTFDdjVyTGJoaEdIeF9YQUY0ZnI2dXN0SG5oNHpscFJjSmtnOUdLNHFBeDBud3NDOVlIOHBSOVBzLUROMTA1cS1aOWgtUQ?oc=5) — *Google News / WCAX*
- `11/11 10:30` [Petrobras Q3 Profit Rises 22% as Oil Production Hits New Record](https://news.google.com/rss/articles/CBMiowFBVV95cUxPQUgwYUttdkNpV2g1ek1TUVdSYUhNNEd2SWt6amM1bVdqSXd5WnhVUTZwTV94REFCbW1DdEV5dWJlbU1RRVI3SjZIMkdEOTBWbFdzV014R1Uxem1XTVRmQUFYN3V3TGhsSkZSc0hrWlZ5YzlYRHhiZUZ3Wl9VUlpFLU8wbk41MVg4TzkwbnlZelNZOTlsMjhGa1NTQi1vYTgxWHZN0gGoAUFVX3lxTE52X21jeUFWQnNGakh5SVRGQUgyblhoQ0ZtQjRneVdhN0paaHVKVDJxVkgtTjNIZmR2Mjlwd0JMQzY4d0R4Y29IdzRvX0RlYjFJQml5dWRxNTJHYVM4bng2MGZFcE55Q3ZPSnpqNVVocUFqNnhWQ2xNbWVYVlU5X0JBUHBra19iQUl1cVpOQVMtQkZhNENwRVluWHFtRVBFNmhNT1IzMVVoSw?oc=5) — *Google News / Crude Oil Prices Today | OilPrice.com*
- `11/11 10:17` [Oil prices rise from 2-mth low with focus on U.S.-Iran peace, inventories](https://news.google.com/rss/articles/CBMiwgFBVV95cUxOejg0VEZTNjhxVVNzODZoM096SndDUXRoVGhHblRFVG8xSEUtNFA5VDl2bFR1dEZ4UVpoUmI4SWRHWE54bmN5ZmFlVTJqU1BGRldDcUtFR1d4Mmx5amRTUVROX1dDS1ZYLVpnZFF2R1FkUklERWpmeUV2LXFzb0RCV0hDYjNoS0VMYnpqOHZwOTU1b1ZNZUtjYlIzZHN0UUxfbWVjMHFyTDVvbHhrNktGQVZOUFBCWUpUZlN0VlJmSEg4UQ?oc=5) — *Google News / Investing.com*

### 中文（鉅亨網美股）
- `11/11 11:34` [上週才說快觸線、這週就確認 Anthropic上市前宣布AI已觸發安全警報](https://news.cnyes.com/news/id/6657917) — *鉅亨網*：Anthropic確認已觸發「負責任擴展政策」安全門檻，將增設防護措施並評估政策調整。&#10;公司周二啟動IPO路演，公開說明書訂每股70至78美元，最高估值約1.4兆美元，最快11月19日掛牌。&#10;輝達、Google各將買10億美元股份，文件並警告災難性AI事件損失恐連整個保險業都難承擔。
- `11/11 11:30` [OpenAI開放商家在ChatGPT設AI代理 12億用戶成商家廣告新入口](https://news.cnyes.com/news/id/6657929) — *鉅亨網*：OpenAI推「Businesses in ChatGPT」測試版，企業可認領檔案、上架商品資訊並用AI代理回覆消費者。&#10;Business Hub可驗證商家、更新定價與庫存、查看瀏覽與點擊等彙總洞察，首波開放美商與線上服務業者。&#10;商家可贊助Business Agent作為廣告落地點；不分享用戶對話。
- `11/11 11:20` [Anthropic IPO估值上看1.4兆美元！高盛估2030年營收達6850億美元、7名創辦人套現17.5億美元](https://news.cnyes.com/news/id/6657968) — *鉅亨網*：Anthropic啟動IPO路演，目標估值1.4兆美元，預計11月18日定價、隔日交易，募資規模達904億美元。&#10;高盛估Anthropic營收2030年達6850億美元，2027年營運虧損恐擴大至404億美元。&#10;7名共同創辦人擬在IPO合計套現17.5億美元；高盛稱算力支出將近4000億美元。
- `11/11 11:20` [輝達跌不停、晶片股蒸發近5兆美元 美銀示警標普500本身風險更大](https://news.cnyes.com/news/id/6657920) — *鉅亨網*：美銀薩布拉曼尼安示警標普500指數風險高，稱全球最擁擠交易標的是標普指數本身。&#10;10月CPI高於預期後，標普500期貨5分鐘跌約0.5%，5年期美債殖利率創9月來最大升幅，10年期一度破5%。&#10;輝達周二重挫約6%，晶片股自10月6日市值蒸發近5兆美元；標普500等權重指數同期漲約4%。
- `11/11 11:10` [頂級富豪婚姻 一窺魏哲家與妻子牛慶容低調婚姻](https://news.cnyes.com/news/id/6636937) — *鉅亨網*：魏哲家早年赴台工作住台積電園區宿舍，家人留美；蔣尚義回憶兩人同棟宿舍、家人都在美國&#10;2021/10/6魏哲家贈牛慶容1,000張台積電股票，實際100萬股、市值5.71億元；兩個多月後夫妻再贈3子各100張&#10;2025/11/8台積電運動會上，牛慶容少見與魏哲家同框；魏哲家也感謝家人支持，婚姻生活低調
- `11/11 10:53` [Anthropic IPO未定價先現溢價！投資人提前轉售配售份額、買家須加價3%至5%](https://news.cnyes.com/news/id/6657930) — *鉅亨網*：Anthropic IPO未定價，部分投資人提前轉售配售份額，開價較發行價高3%至5%&#10;JonesTrading代表部分投資人推介最高約5億美元配售份額；IPO規模最高募資644億美元&#10;Anthropic預計11/18定價、11/19掛牌；第三季營收169.56億美元，GAAP營業虧損5.67億美元
- `11/11 10:41` [黃仁勳坦承外遇後 CNBC：已回輝達總部上班 內部氣氛悲傷](https://news.cnyes.com/news/id/6658151) — *鉅亨網*：知情人士透露，黃仁勳周一走進加州聖克拉拉總部上班，如往常般主持高階主管晨會。&#10;黃仁勳周二也進辦公室開會。&#10;輝達內部人士表示，黃仁勳試圖營造「一切如常」的氛圍，但公司內部氣氛悲傷。
- `11/11 10:20` [川普稱川習會前不知黃仁勳婚外情 《紐時》揭7個月內至少兩度獲簡報](https://news.cnyes.com/news/id/6658152) — *鉅亨網*：紐時指川普今年5月赴中前已聽取黃仁勳涉中國情報疑慮簡報，非他所稱不知情&#10;黃仁勳坦承與Megaspeed前主管黃樂婚外情；輝達稱已啟動獨立調查，暫無國安或機密受損跡象&#10;分析師稱黃仁勳對華府遊說影響力減弱，Blackwell對中出口許可恐被更嚴審查
